# 03 â€” Feature Engineering â€” Final ML-Ready Dataset
Historical ML feature engineering for SIH Hyperlocal Monsoon prototype.

**Study area:** Sangrur District, 6 legacy Bhuvan blocks â€” Dhuri, Lehra, Malerkotla, Moonak, Sangrur, Sunam  
**Authoritative boundaries:** `data/raw/boundaries/sangrur_blocks_bhuvan.gpkg` (layer `sangrur_blocks`)  
**Input:** `data/processed/forecast_features_rainfall.parquet` (from Notebook 02 Cells 36-40)  
**Output (eventual):** `data/processed/historical_ml_dataset.parquet` (for Notebook 04)

**This notebook transforms the validated historical forecast/rainfall dataset into the FINAL ML-ready feature dataset.**
Do NOT train models here. Do NOT evaluate yet. Cells 1-5 only for now: imports, load, audit, feature groups, clean frame.


In [1]:
# Cell 1 â€” Imports and project setup
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
# rasterio only if soil processing later
try:
    import rasterio
    _has_rasterio = True
except ImportError:
    _has_rasterio = False

# Robust project-root detection (as in 01/02)
CWD = Path.cwd().resolve()
if CWD.name == "notebooks":
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = Path("..").resolve()
    if not (PROJECT_ROOT / "notebooks").exists() and not (PROJECT_ROOT / "data").exists():
        PROJECT_ROOT = Path.cwd().resolve()
        if PROJECT_ROOT.name == "notebooks":
            PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_CLIMATE = DATA_RAW / "climate"
DATA_SOIL = DATA_RAW / "soil"
DATA_BOUNDARIES = DATA_RAW / "boundaries"

print(f"PROJECT_ROOT: {PROJECT_ROOT.resolve()}")
print(f"DATA_RAW: {DATA_RAW.resolve()}")
print(f"DATA_PROCESSED: {DATA_PROCESSED.resolve()}")
print(f"DATA_CLIMATE: {DATA_CLIMATE.resolve()} (exists {DATA_CLIMATE.exists()})")
print(f"DATA_SOIL: {DATA_SOIL.resolve()} (exists {DATA_SOIL.exists()})")
print(f"DATA_BOUNDARIES: {DATA_BOUNDARIES.resolve()}")
import sys, platform
print(f"\nPython: {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | geopandas {gpd.__version__}")
if _has_rasterio:
    print(f"rasterio {rasterio.__version__}")
else:
    print("rasterio not available (will be needed for soil later)")
print(f"\nProcessed-data dir: {DATA_PROCESSED.resolve()} ({len(list(DATA_PROCESSED.glob('*')))} entries)")
for p in sorted(DATA_PROCESSED.glob("*.parquet")):
    print(f"  parquet: {p.name} ({p.stat().st_size/1024:.1f} KB)")
for p in sorted(DATA_PROCESSED.glob("*.csv")):
    print(f"  csv: {p.name} ({p.stat().st_size/1024:.1f} KB)")
# Do not load datasets yet


PROJECT_ROOT: C:\Users\Swarnim\Desktop\ML projects\saarthi-2
DATA_RAW: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw
DATA_PROCESSED: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed
DATA_CLIMATE: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\climate (exists True)
DATA_SOIL: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\soil (exists True)
DATA_BOUNDARIES: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\boundaries

Python: 3.11.9 | pandas 2.3.3 | numpy 2.4.1 | geopandas 1.1.4
rasterio 1.4.4

Processed-data dir: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed (22 entries)
  parquet: final_ml_dataset.parquet (1203.6 KB)
  parquet: final_ml_dataset_legacy.parquet (20.9 KB)
  parquet: forecast_features_base.parquet (416.8 KB)
  parquet: forecast_features_base_legacy.parquet (6.1 KB)
  parquet: forecast_features_rainfall.parquet (1076.4 KB)
  parquet: forecast_features_rainfall_legacy.parquet (14.7 KB)
  parquet: historical_fore

In [2]:
# Cell 2 â€” Load the Notebook 02 output (forecast_features_rainfall)
from pathlib import Path
import pandas as pd
import numpy as np

# Robust PROJECT (reuse if already defined, else recompute)
try:
    PROJECT_ROOT
except NameError:
    CWD = Path.cwd().resolve()
    if CWD.name == "notebooks":
        PROJECT_ROOT = CWD.parent
    else:
        PROJECT_ROOT = Path("..").resolve()
        if not (PROJECT_ROOT / "notebooks").exists():
            PROJECT_ROOT = Path.cwd().resolve()
    DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

parquet_path = DATA_PROCESSED / "forecast_features_rainfall.parquet"
csv_path = DATA_PROCESSED / "forecast_features_rainfall.csv"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    print(f"Loaded Parquet (preferred): {parquet_path.resolve()} shape {df.shape}")
    src = "parquet"
elif csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"Loaded CSV (fallback): {csv_path.resolve()} shape {df.shape}")
    src = "csv"
else:
    raise FileNotFoundError(f"Neither {parquet_path} nor {csv_path} found â€” run Notebook 02 Cells 36-40 first")

# Normalize
df["forecast_date"] = pd.to_datetime(df["forecast_date"])
df["block"] = df["block"].astype(str).str.strip()
# All numeric rainfall columns
rain_cols = [c for c in df.columns if c.startswith("rain_") or c.startswith("gefs_") or c=="target_7d_rainfall_mm"]
for c in rain_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print(f"\nShape: {df.shape} (rows, cols)")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")
print(f"Date range: {df['forecast_date'].min()} to {df['forecast_date'].max()} ({df['forecast_date'].nunique()} unique forecast dates)")
print(f"Unique blocks: {sorted(df['block'].unique().tolist())} ({df['block'].nunique()})")
print(f"Forecast dates: {sorted(df['forecast_date'].dt.strftime('%Y-%m-%d').unique().tolist())}")
print(f"Source: {src} (Parquet preferred)")

# Expected important columns
expected = [
    "forecast_date","block",
    "gefs_d1","gefs_d2","gefs_d3","gefs_d4","gefs_d5","gefs_d6","gefs_d7",
    "rain_1d","rain_3d","rain_7d","rain_14d","rain_30d",
    "rain_lag_1","rain_lag_2","rain_lag_3","rain_lag_4","rain_lag_5","rain_lag_6","rain_lag_7",
    "target_7d_rainfall_mm"
]
missing = [c for c in expected if c not in df.columns]
present = [c for c in expected if c in df.columns]
print(f"\nExpected columns: {len(expected)}")
print(f"Present: {len(present)} {present}")
if missing:
    print(f"MISSING expected columns: {missing} â€” clearly reported (do NOT create fake zeros)")
else:
    print("All expected columns present â€” OK (do NOT silently create fake columns)")

# Quick missing summary for expected
print("\nMissing per expected column:")
for c in expected:
    if c in df.columns:
        n = df[c].isna().sum()
        print(f"  {c}: {n} missing ({n/len(df)*100:.1f}%)")

# Keep for next cells
DF_03 = df.copy()


Loaded Parquet (preferred): C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed\forecast_features_rainfall.parquet shape (6588, 22)

Shape: (6588, 22) (rows, cols)
Columns (22): ['forecast_date', 'block', 'gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7', 'target_7d_rainfall_mm', 'rain_1d', 'rain_3d', 'rain_7d', 'rain_14d', 'rain_30d', 'rain_lag_1', 'rain_lag_2', 'rain_lag_3', 'rain_lag_4', 'rain_lag_5', 'rain_lag_6', 'rain_lag_7']
Date range: 2016-06-01 00:00:00 to 2025-09-30 00:00:00 (1098 unique forecast dates)
Unique blocks: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'] (6)
Forecast dates: ['2016-06-01', '2016-06-02', '2016-06-03', '2016-06-04', '2016-06-05', '2016-06-06', '2016-06-07', '2016-06-08', '2016-06-09', '2016-06-10', '2016-06-11', '2016-06-12', '2016-06-13', '2016-06-14', '2016-06-15', '2016-06-16', '2016-06-17', '2016-06-18', '2016-06-19', '2016-06-20', '2016-06-21', '2016-06-22', '2016-06-23', '2016-06-24', '2016-06-

In [3]:
# Cell 3 â€” Initial ML dataset audit
import pandas as pd
import numpy as np

if 'DF_03' not in locals():
    raise RuntimeError("DF_03 not found â€” run Cell 2 first")

df = DF_03.copy()
print(f"Auditing DF_03 shape {df.shape}")

# 1-10 checks
print("\n1. One row = one forecast_date + block:")
print(f"  Rows: {len(df)}")

print("\n2. forecast_date + block is unique:")
dup = df.duplicated(subset=["forecast_date","block"]).sum()
print(f"  Duplicates: {dup} (expected 0) -> {'PASS' if dup==0 else 'FAIL'}")

print("\n3. Six expected Sangrur blocks exist:")
expected_blocks = {"Dhuri","Lehra","Malerkotla","Moonak","Sangrur","Sunam"}
found_blocks = set(df["block"].tolist())
print(f"  Expected: {sorted(expected_blocks)}")
print(f"  Found: {sorted(found_blocks)} -> {'PASS' if found_blocks==expected_blocks else 'FAIL'}")

print("\n4. Dates are chronological:")
is_sorted = df["forecast_date"].is_monotonic_increasing or df.sort_values(["forecast_date","block"])["forecast_date"].is_monotonic_increasing
# Check sorted order
sorted_df = df.sort_values(["forecast_date","block"])
is_chrono = sorted_df["forecast_date"].is_monotonic_increasing
print(f"  Chronological (after sort): {is_chrono} -> {'PASS' if is_chrono else 'FAIL'}")
print(f"  Date range: {df['forecast_date'].min()} to {df['forecast_date'].max()}")

print("\n5. target_7d_rainfall_mm exists:")
has_target = "target_7d_rainfall_mm" in df.columns
print(f"  Exists: {has_target} -> {'PASS' if has_target else 'FAIL'}")

print("\n6. Target contains no negative values:")
if has_target:
    neg = (df["target_7d_rainfall_mm"] < 0).sum()
    print(f"  Negative: {neg} -> {'PASS' if neg==0 else 'FAIL'}")
else:
    print("  Skipped (no target)")

print("\n7. Target contains no infinite values:")
if has_target:
    inf = np.isinf(pd.to_numeric(df["target_7d_rainfall_mm"], errors="coerce")).sum()
    print(f"  Infinite: {inf} -> {'PASS' if inf==0 else 'FAIL'}")

print("\n8. GEFS features are numeric:")
for c in [f"gefs_d{i}" for i in range(1,8)]:
    if c in df.columns:
        is_num = pd.api.types.is_numeric_dtype(df[c])
        print(f"  {c}: numeric {is_num} -> {'PASS' if is_num else 'FAIL'}")

print("\n9. Recent rainfall features are numeric:")
for c in ["rain_1d","rain_3d","rain_7d","rain_14d","rain_30d"] + [f"rain_lag_{i}" for i in range(1,8)]:
    if c in df.columns:
        is_num = pd.api.types.is_numeric_dtype(df[c])
        print(f"  {c}: numeric {is_num} -> {'PASS' if is_num else 'FAIL'}")

print("\n10. No obvious duplicate records exist:")
print(f"  Duplicates (forecast_date+block): {dup} -> {'PASS' if dup==0 else 'FAIL'}")

# Compact audit table
print("\n=== Audit table (column, dtype, missing, missing%, min, max, mean) ===")
rows = []
for col in df.columns:
    dtype = str(df[col].dtype)
    missing = df[col].isna().sum()
    pct = missing/len(df)*100
    # For numeric, compute min/max/mean
    if pd.api.types.is_numeric_dtype(df[col]):
        mn = df[col].min()
        mx = df[col].max()
        mean = df[col].mean()
    else:
        mn = mx = mean = np.nan
    rows.append({"column": col, "dtype": dtype, "missing_count": missing, "missing_percent": round(pct,1), "min": mn, "max": mx, "mean": mean})
audit = pd.DataFrame(rows)
# Do not print huge amounts â€” compact
print(audit.to_string(index=False))

print(f"\nSummary:")
print(f"  total rows: {len(df)}")
print(f"  total forecast dates: {df['forecast_date'].nunique()}")
print(f"  total blocks: {df['block'].nunique()}")
print(f"  duplicate count: {dup}")
if has_target:
    print(f"  target missing count: {df['target_7d_rainfall_mm'].isna().sum()}")
    print(f"  target invalid (negative/inf): {(df['target_7d_rainfall_mm']<0).sum()} negative, {np.isinf(pd.to_numeric(df['target_7d_rainfall_mm'], errors='coerce')).sum()} inf")


Auditing DF_03 shape (6588, 22)

1. One row = one forecast_date + block:
  Rows: 6588

2. forecast_date + block is unique:
  Duplicates: 0 (expected 0) -> PASS

3. Six expected Sangrur blocks exist:
  Expected: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']
  Found: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'] -> PASS

4. Dates are chronological:
  Chronological (after sort): True -> PASS
  Date range: 2016-06-01 00:00:00 to 2025-09-30 00:00:00

5. target_7d_rainfall_mm exists:
  Exists: True -> PASS

6. Target contains no negative values:
  Negative: 0 -> PASS

7. Target contains no infinite values:
  Infinite: 0 -> PASS

8. GEFS features are numeric:
  gefs_d1: numeric True -> PASS
  gefs_d2: numeric True -> PASS
  gefs_d3: numeric True -> PASS
  gefs_d4: numeric True -> PASS
  gefs_d5: numeric True -> PASS
  gefs_d6: numeric True -> PASS
  gefs_d7: numeric True -> PASS

9. Recent rainfall features are numeric:
  rain_1d: numeric True -> PASS
  rain_

In [4]:
# Cell 4 â€” Define feature groups
# Explicit lists for final ML dataset â€” makes feature construction transparent

GEFS_FEATURES = [
    "gefs_d1",
    "gefs_d2",
    "gefs_d3",
    "gefs_d4",
    "gefs_d5",
    "gefs_d6",
    "gefs_d7",
]

RECENT_RAINFALL_FEATURES = [
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "rain_14d",
    "rain_30d",
    "rain_lag_1",
    "rain_lag_2",
    "rain_lag_3",
    "rain_lag_4",
    "rain_lag_5",
    "rain_lag_6",
    "rain_lag_7",
]

TARGET_COLUMNS = [
    "target_7d_rainfall_mm"
]

# Planned â€” to be populated in later cells (do NOT add yet)
ENSO_FEATURES = []        # e.g., ["ENSO_index", "ENSO_category"] â€” Cell 6+
SEASONAL_FEATURES = []    # e.g., ["sin_doy", "cos_doy"] â€” Cell 7+
SOIL_FEATURES = []        # e.g., ["soil_clay","soil_sand","soil_silt","soil_soc","soil_ph"] â€” Cell 8+
SPATIAL_FEATURES = []     # e.g., ["latitude","longitude"] â€” Cell 9+

IDENTIFIER_COLUMNS = [
    "forecast_date",
    "block"
]

print("Feature groups defined explicitly (not auto-fed):")
print(f"  IDENTIFIER_COLUMNS ({len(IDENTIFIER_COLUMNS)}): {IDENTIFIER_COLUMNS}")
print(f"  GEFS_FEATURES ({len(GEFS_FEATURES)}): {GEFS_FEATURES}")
print(f"  RECENT_RAINFALL_FEATURES ({len(RECENT_RAINFALL_FEATURES)}): {RECENT_RAINFALL_FEATURES}")
print(f"  TARGET_COLUMNS ({len(TARGET_COLUMNS)}): {TARGET_COLUMNS}")
print(f"  ENSO_FEATURES ({len(ENSO_FEATURES)}): {ENSO_FEATURES} â€” planned for later")
print(f"  SEASONAL_FEATURES ({len(SEASONAL_FEATURES)}): {SEASONAL_FEATURES} â€” planned")
print(f"  SOIL_FEATURES ({len(SOIL_FEATURES)}): {SOIL_FEATURES} â€” planned")
print(f"  SPATIAL_FEATURES ({len(SPATIAL_FEATURES)}): {SPATIAL_FEATURES} â€” planned")

# Verify that current DF has these columns
if 'DF_03' in locals():
    missing_gefs = [c for c in GEFS_FEATURES if c not in DF_03.columns]
    missing_rain = [c for c in RECENT_RAINFALL_FEATURES if c not in DF_03.columns]
    missing_target = [c for c in TARGET_COLUMNS if c not in DF_03.columns]
    if missing_gefs: print(f"  WARNING GEFS missing in DF_03: {missing_gefs}")
    if missing_rain: print(f"  WARNING rainfall missing in DF_03: {missing_rain}")
    if missing_target: print(f"  WARNING target missing in DF_03: {missing_target}")
    if not missing_gefs and not missing_rain and not missing_target:
        print("  All current groups present in DF_03 â€” OK")
else:
    print("  DF_03 not found â€” cannot verify presence")

# For later: final input feature list will be GEFS + RAIN + ENSO + SEASONAL + SOIL + SPATIAL
ALL_PLANNED_INPUTS = GEFS_FEATURES + RECENT_RAINFALL_FEATURES + ENSO_FEATURES + SEASONAL_FEATURES + SOIL_FEATURES + SPATIAL_FEATURES
print(f"\nCurrently available inputs: {len(GEFS_FEATURES + RECENT_RAINFALL_FEATURES)} (GEFS+rainfall only, ENSO/soil/spatial/seasonal pending)")


Feature groups defined explicitly (not auto-fed):
  IDENTIFIER_COLUMNS (2): ['forecast_date', 'block']
  GEFS_FEATURES (7): ['gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7']
  RECENT_RAINFALL_FEATURES (12): ['rain_1d', 'rain_3d', 'rain_7d', 'rain_14d', 'rain_30d', 'rain_lag_1', 'rain_lag_2', 'rain_lag_3', 'rain_lag_4', 'rain_lag_5', 'rain_lag_6', 'rain_lag_7']
  TARGET_COLUMNS (1): ['target_7d_rainfall_mm']
  ENSO_FEATURES (0): [] â€” planned for later
  SEASONAL_FEATURES (0): [] â€” planned
  SOIL_FEATURES (0): [] â€” planned
  SPATIAL_FEATURES (0): [] â€” planned
  All current groups present in DF_03 â€” OK

Currently available inputs: 19 (GEFS+rainfall only, ENSO/soil/spatial/seasonal pending)


In [5]:
# Cell 5 â€” Create a clean feature-building frame
import pandas as pd

if 'DF_03' not in locals():
    raise RuntimeError("DF_03 not found â€” run Cell 2")
if 'GEFS_FEATURES' not in locals() or 'RECENT_RAINFALL_FEATURES' not in locals():
    raise RuntimeError("Feature groups not defined â€” run Cell 4")

# Start from validated DF_03
df = DF_03.copy()
df["forecast_date"] = pd.to_datetime(df["forecast_date"])
df["block"] = df["block"].astype(str).str.strip()

# Retain only current groups + target (do NOT add ENSO/soil/spatial/calendar yet)
keep_cols = IDENTIFIER_COLUMNS + GEFS_FEATURES + RECENT_RAINFALL_FEATURES + TARGET_COLUMNS
# Filter to those that actually exist (do NOT create fake zeros for missing)
available_keep = [c for c in keep_cols if c in df.columns]
missing_keep = [c for c in keep_cols if c not in df.columns]
if missing_keep:
    print(f"Note: expected cols missing (do NOT create fake zeros): {missing_keep}")

ml_features = df[available_keep].copy()
ml_features = ml_features.sort_values(["forecast_date","block"]).reset_index(drop=True)

# Preserve target separately from input features
# X_BASE contains only current input features (GEFS + recent rainfall)
X_BASE_COLS = [c for c in (GEFS_FEATURES + RECENT_RAINFALL_FEATURES) if c in ml_features.columns]
X_BASE = ml_features[X_BASE_COLS].copy()
y = ml_features[TARGET_COLUMNS[0]].copy() if TARGET_COLUMNS[0] in ml_features.columns else pd.Series(dtype=float)

print(f"ml_features shape: {ml_features.shape} (one row = one forecast_date + block)")
print(f"X_BASE shape: {X_BASE.shape} (inputs: GEFS {len([c for c in GEFS_FEATURES if c in X_BASE.columns])} + rainfall {len([c for c in RECENT_RAINFALL_FEATURES if c in X_BASE.columns])} = {len(X_BASE_COLS)} cols)")
print(f"y shape: {y.shape} (target: {TARGET_COLUMNS[0]})")
print(f"\nX_BASE columns ({len(X_BASE.columns)}): {X_BASE.columns.tolist()}")
print(f"Target column: {TARGET_COLUMNS[0]}")

# Missing-value summary â€” all features expected complete on JJAS>=2016 rebuild
print("\nMissing-value summary (keep NaNs explicit, do NOT fill with zero):")
for col in X_BASE.columns:
    n = X_BASE[col].isna().sum()
    print(f"  {col}: {n} missing ({n/len(X_BASE)*100:.1f}%)")
print(f"  {TARGET_COLUMNS[0]}: {y.isna().sum()} missing ({y.isna().mean()*100:.1f}%)")

# Important notes
print("\nNotes:")
print("  - Do NOT drop rows merely because ENSO/soil/spatial/calendar not yet added")
print("  - all 7 GEFS leads present (corrected mapping: folder D, files D+1..D+7)")
print("  - y is target_7d_rainfall_mm (CHIRPS D+1..D+7), NOT an input feature")
print("  - Chronological order preserved, no shuffle")

# Keep for next notebook steps (will be extended in Cells 6+)
ML_FEATURES = ml_features.copy()
X_BASE_03 = X_BASE.copy()
Y_03 = y.copy()

# Preview
print("\nPreview ml_features:")
print(ml_features.head(3).to_string(index=False))


ml_features shape: (6588, 22) (one row = one forecast_date + block)
X_BASE shape: (6588, 19) (inputs: GEFS 7 + rainfall 12 = 19 cols)
y shape: (6588,) (target: target_7d_rainfall_mm)

X_BASE columns (19): ['gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7', 'rain_1d', 'rain_3d', 'rain_7d', 'rain_14d', 'rain_30d', 'rain_lag_1', 'rain_lag_2', 'rain_lag_3', 'rain_lag_4', 'rain_lag_5', 'rain_lag_6', 'rain_lag_7']
Target column: target_7d_rainfall_mm

Missing-value summary (keep NaNs explicit, do NOT fill with zero):
  gefs_d1: 0 missing (0.0%)
  gefs_d2: 0 missing (0.0%)
  gefs_d3: 0 missing (0.0%)
  gefs_d4: 0 missing (0.0%)
  gefs_d5: 0 missing (0.0%)
  gefs_d6: 0 missing (0.0%)
  gefs_d7: 0 missing (0.0%)
  rain_1d: 0 missing (0.0%)
  rain_3d: 0 missing (0.0%)
  rain_7d: 0 missing (0.0%)
  rain_14d: 0 missing (0.0%)
  rain_30d: 0 missing (0.0%)
  rain_lag_1: 0 missing (0.0%)
  rain_lag_2: 0 missing (0.0%)
  rain_lag_3: 0 missing (0.0%)
  rain_lag_4: 0 missing (0

In [6]:
# Cell 6 â€” Discover and load ENSO data
from pathlib import Path
import pandas as pd
import numpy as np
import requests

try:
    PROJECT_ROOT
except NameError:
    CWD = Path.cwd().resolve()
    PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else Path("..").resolve()
    DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_CLIMATE = PROJECT_ROOT / "data" / "raw" / "climate"
DATA_CLIMATE.mkdir(parents=True, exist_ok=True)

print("Inspecting data/raw/climate/:")
existing = sorted(DATA_CLIMATE.glob("*"))
print(f"  entries: {len(existing)}")
for p in existing:
    print(f"    {p.name} ({p.stat().st_size/1024:.1f} KB)")
if not existing:
    print("  (empty â€” no ENSO dataset exists locally yet)")

# Documented source (docs/project_context/03_DATASETS.md Â§3):
# NOAA CPC ERSSTv5 Nino indices. The 81-10 file is FROZEN at 2020-12
# (verified: last row 2020-12) and cannot cover forecast dates up to 2025-08-01.
# Use the maintained 91-20 file from the SAME product family (same format/columns).
ENSO_URL = "https://www.cpc.ncep.noaa.gov/data/indices/ersst5.nino.mth.91-20.ascii"
ENSO_FILE = DATA_CLIMATE / "ersst5.nino.mth.91-20.ascii"
print(f"\nDocumented source: {ENSO_URL}")
print("Note: docs list ersst5.nino.mth.81-10.ascii, but it ends 2020-12;")
print("91-20 is the same NOAA CPC ERSSTv5 product, updated to present. Documented here, not invented.")

if ENSO_FILE.exists() and ENSO_FILE.stat().st_size > 1000:
    print(f"Already exists: {ENSO_FILE.name} ({ENSO_FILE.stat().st_size/1024:.1f} KB) â€” skipping download")
else:
    print(f"Downloading to {ENSO_FILE.resolve()} ...")
    r = requests.get(ENSO_URL, timeout=60)
    r.raise_for_status()
    tmp = ENSO_FILE.with_suffix(".ascii.tmp")
    tmp.write_bytes(r.content)
    tmp.rename(ENSO_FILE)
    print(f"Downloaded {len(r.content)/1024:.1f} KB -> {ENSO_FILE.name}")

# Inspect raw file structure before parsing
lines = ENSO_FILE.read_text().splitlines()
print(f"\nRaw file: {ENSO_FILE.name}, {len(lines)} lines")
print(f"Header: {lines[0]!r}")
print(f"First data: {lines[1]!r}")
print(f"Last data:  {lines[-1]!r}")

# Columns (positional â€” header has duplicate 'ANOM' labels, so assign explicit names)
ENSO_COLS = ["YR","MON","NINO12","NINO12_ANOM","NINO3","NINO3_ANOM",
             "NINO4","NINO4_ANOM","NINO34","NINO34_ANOM"]
ENSO_RAW = pd.read_csv(ENSO_FILE, sep=r"\s+", names=ENSO_COLS, skiprows=1)
print(f"\nParsed shape: {ENSO_RAW.shape} (rows, cols)")
print(f"Columns: {ENSO_RAW.columns.tolist()}")
print(f"Temporal resolution: MONTHLY (one row = one calendar month)")
first, last = ENSO_RAW.iloc[0], ENSO_RAW.iloc[-1]
print(f"Date coverage: {int(first['YR'])}-{int(first['MON']):02d} to "
      f"{int(last['YR'])}-{int(last['MON']):02d} ({len(ENSO_RAW)} months)")
print(f"\nAvailable ENSO variables: NINO12/NINO3/NINO4/NINO34 SST + *_ANOM anomalies")
print("Missing-value counts per column:")
for c in ENSO_COLS:
    n = ENSO_RAW[c].isna().sum()
    print(f"  {c}: {n} missing ({n/len(ENSO_RAW)*100:.1f}%)")

# Primary index decision (docs 03_DATASETS Â§3 + 04_ML_STRATEGY: one primary index first, e.g. Nino3.4/ONI)
print("\nPrimary index decision: NINO34_ANOM (Nino3.4 SSTA, 1991-2020 climatology).")
print("Reason: documented project decision (one primary numeric index first; Nino3.4/ONI).")
print("NOT using NINO12/NINO3/NINO4 or raw SST â€” single index only, no redundancy.")
print("Anomaly base period (91-20) differs from frozen 81-10 file â€” consistent within this file, noted.")

# Quick look at the primary series around our forecast era
tmpd = ENSO_RAW.copy()
tmpd["ym"] = tmpd["YR"].astype(int).astype(str) + "-" + tmpd["MON"].astype(int).astype(str).str.zfill(2)
print("\nNINO34_ANOM recent sample:")
print(tmpd[tmpd["YR"] >= 2024][["YR","MON","NINO34_ANOM"]].to_string(index=False))


Inspecting data/raw/climate/:
  entries: 1
    ersst5.nino.mth.91-20.ascii (65.5 KB)

Documented source: https://www.cpc.ncep.noaa.gov/data/indices/ersst5.nino.mth.91-20.ascii
Note: docs list ersst5.nino.mth.81-10.ascii, but it ends 2020-12;
91-20 is the same NOAA CPC ERSSTv5 product, updated to present. Documented here, not invented.
Already exists: ersst5.nino.mth.91-20.ascii (65.5 KB) â€” skipping download

Raw file: ersst5.nino.mth.91-20.ascii, 919 lines
Header: ' YR   MON  NINO1+2  ANOM   NINO3    ANOM   NINO4    ANOM   NINO3.4  ANOM'
First data: '1950   1   23.01   -1.55   23.56   -2.10   26.94   -1.38   24.55   -1.99'
Last data:  '2026   6   25.94    2.82   28.33    1.71   30.19    1.22   29.17    1.44'

Parsed shape: (918, 10) (rows, cols)
Columns: ['YR', 'MON', 'NINO12', 'NINO12_ANOM', 'NINO3', 'NINO3_ANOM', 'NINO4', 'NINO4_ANOM', 'NINO34', 'NINO34_ANOM']
Temporal resolution: MONTHLY (one row = one calendar month)
Date coverage: 1950-01 to 2026-06 (918 months)

Available ENSO 

In [7]:
# Cell 7 â€” Clean and standardize the ENSO series
import pandas as pd
import numpy as np

if 'ENSO_RAW' not in locals():
    raise RuntimeError("ENSO_RAW not found â€” run Cell 6 first")

df = ENSO_RAW.copy()

# Build clean frame: enso_date (first day of month) + enso_value (Nino3.4 anomaly)
enso = pd.DataFrame({
    "enso_date": pd.to_datetime(dict(year=df["YR"].astype(int),
                                     month=df["MON"].astype(int), day=1)),
    "enso_value": pd.to_numeric(df["NINO34_ANOM"], errors="coerce"),
})

# Sentinel check: CPC uses -99.99 for missing trailing months â€” convert to NaN explicitly
n_sentinel = int((df["NINO34_ANOM"] == -99.99).sum())
if n_sentinel:
    enso.loc[enso["enso_value"] == -99.99, "enso_value"] = np.nan
print(f"Sentinel -99.99 values converted to NaN: {n_sentinel}")

# Duplicate-date check â€” report conflicts, never silently choose
dup_mask = enso.duplicated(subset=["enso_date"], keep=False)
n_dup = int(dup_mask.sum())
print(f"Duplicate enso_date rows: {n_dup}")
if n_dup:
    conf = enso[dup_mask].sort_values("enso_date")
    print(conf.to_string(index=False))
    # Conflict = same date, different values?
    g = conf.groupby("enso_date")["enso_value"].nunique()
    conflicts = g[g > 1]
    if len(conflicts):
        print(f"CONFLICTING duplicate dates (STOP, do not proceed): {conflicts.index.tolist()}")
        raise ValueError("Conflicting duplicate ENSO dates â€” manual resolution required")
    enso = enso.drop_duplicates(subset=["enso_date"], keep="first")
    print("Exact duplicates removed (identical values, safe).")
else:
    print("No duplicate dates â€” OK")

# Validity checks
print(f"\nNaN enso_value: {int(enso['enso_value'].isna().sum())}")
print(f"Infinite enso_value: {int(np.isinf(enso['enso_value'].dropna()).sum())}")
print(f"Non-numeric coerced: {int(enso['enso_value'].isna().sum())} (includes sentinels)")
print(f"Date ordering OK: {bool(enso['enso_date'].is_monotonic_increasing)}")

enso = enso.sort_values("enso_date").reset_index(drop=True)
ENSO_CLEAN = enso.copy()
print(f"\n=== Clean ENSO summary ===")
print(f"  earliest: {ENSO_CLEAN['enso_date'].min().date()}")
print(f"  latest:   {ENSO_CLEAN['enso_date'].max().date()}")
print(f"  records:  {len(ENSO_CLEAN)} months")
print(f"  missing values: {int(ENSO_CLEAN['enso_value'].isna().sum())}")
print(f"  duplicate dates: {int(ENSO_CLEAN.duplicated('enso_date').sum())}")
print(f"  min: {ENSO_CLEAN['enso_value'].min():.2f}  max: {ENSO_CLEAN['enso_value'].max():.2f}  "
      f"median: {ENSO_CLEAN['enso_value'].median():.2f}")
print("No invention/interpolation applied â€” raw monthly values only. Raw file untouched.")


Sentinel -99.99 values converted to NaN: 0
Duplicate enso_date rows: 0
No duplicate dates â€” OK

NaN enso_value: 0
Infinite enso_value: 0
Non-numeric coerced: 0 (includes sentinels)
Date ordering OK: True

=== Clean ENSO summary ===
  earliest: 1950-01-01
  latest:   2026-06-01
  records:  918 months
  missing values: 0
  duplicate dates: 0
  min: -2.45  max: 2.72  median: -0.25
No invention/interpolation applied â€” raw monthly values only. Raw file untouched.


In [8]:
# Cell 8 â€” Determine temporal alignment method
import pandas as pd
import numpy as np
try:
    from IPython.display import display, Markdown
    display(Markdown(
'''# Cell 8 â€” ENSO temporal alignment (as-of join)

**ENSO source frequency: MONTHLY** (one row = one calendar month, dated 1st of month).

**Alignment rule:** for forecast issue date **D**, attach the monthly ENSO value of the
**previous calendar month** â€” i.e. `asof_month = month(D) âˆ’ 1 month`.

**Why this avoids future information:**
- The current-month value is *incomplete while month D is in progress* and is published by
  NOAA CPC only after month-end â€” using it at D would leak the future.
- The previous calendar month is *complete* before D and published by early current month,
  so it represents information available at or before D (per `05_DATA_LEAKAGE_RULES.md`:
  `ENSO_index` for `2019-08` attached to `D=2019-09-04`, never `2019-10`).
- Docs (`03_DATASETS.md Â§3`) prescribe joining by year-month of D using the value available
  at D; previous-month is the conservative, always-valid choice of the documented
  `2019-08 or 2019-09` options. Publication-lag edge (first days of month) is negligible
  for a slow monthly climate index and is noted, not hidden.

**Implementation:** reusable `get_enso_asof_date(D)` + vectorized merge on year-month key.
Guarantee: `selected_enso_date <= D` for every row (validated in Cell 9).
'''))
except Exception:
    print("MONTHLY ENSO -> attach previous calendar month value (see docstring).")

if 'ENSO_CLEAN' not in locals():
    raise RuntimeError("ENSO_CLEAN not found â€” run Cell 7 first")

# Lookup: (year, month) -> (enso_date, enso_value)
_ENSO_LUT = {(int(r["enso_date"].year), int(r["enso_date"].month)): (r["enso_date"], r["enso_value"])
             for _, r in ENSO_CLEAN.iterrows()}

def get_enso_asof_date(forecast_date):
    '''Return (enso_date, enso_value) available at forecast issue date D.

    Uses the previous calendar month value. enso_date is the 1st of that
    month, so enso_date <= D always holds. Returns (NaT, NaN) if the
    previous month is absent from the series (reported, never filled).
    '''
    D = pd.Timestamp(forecast_date)
    first_this = pd.Timestamp(D.year, D.month, 1)
    asof = first_this - pd.offsets.MonthBegin(1)  # 1st of previous month
    key = (asof.year, asof.month)
    if key in _ENSO_LUT:
        return _ENSO_LUT[key]
    return (pd.NaT, np.nan)

def add_enso_asof_keys(dates):
    '''Vectorized: DatetimeIndex/Series of D -> DataFrame[asof_enso_date].'''
    D = pd.to_datetime(pd.Series(dates)).reset_index(drop=True)
    first_this = D.dt.to_period("M").dt.to_timestamp()
    asof = first_this - pd.offsets.MonthBegin(1)
    return pd.DataFrame({"forecast_date": D, "asof_enso_date": asof})

# Sanity demo on fixed examples (not dataset-dependent)
for demo in ["2019-09-04", "2010-01-01", "2025-08-01"]:
    ed, ev = get_enso_asof_date(demo)
    print(f"D={demo} -> enso_date={ed.date() if pd.notna(ed) else None}, "
          f"value={ev:.2f} (enso_date <= D: {ed <= pd.Timestamp(demo)})")

print("\nFunction get_enso_asof_date + vectorized add_enso_asof_keys ready.")


# Cell 8 â€” ENSO temporal alignment (as-of join)

**ENSO source frequency: MONTHLY** (one row = one calendar month, dated 1st of month).

**Alignment rule:** for forecast issue date **D**, attach the monthly ENSO value of the
**previous calendar month** â€” i.e. `asof_month = month(D) âˆ’ 1 month`.

**Why this avoids future information:**
- The current-month value is *incomplete while month D is in progress* and is published by
  NOAA CPC only after month-end â€” using it at D would leak the future.
- The previous calendar month is *complete* before D and published by early current month,
  so it represents information available at or before D (per `05_DATA_LEAKAGE_RULES.md`:
  `ENSO_index` for `2019-08` attached to `D=2019-09-04`, never `2019-10`).
- Docs (`03_DATASETS.md Â§3`) prescribe joining by year-month of D using the value available
  at D; previous-month is the conservative, always-valid choice of the documented
  `2019-08 or 2019-09` options. Publication-lag edge (first days of month) is negligible
  for a slow monthly climate index and is noted, not hidden.

**Implementation:** reusable `get_enso_asof_date(D)` + vectorized merge on year-month key.
Guarantee: `selected_enso_date <= D` for every row (validated in Cell 9).


D=2019-09-04 -> enso_date=2019-08-01, value=0.04 (enso_date <= D: True)
D=2010-01-01 -> enso_date=2009-12-01, value=1.74 (enso_date <= D: True)
D=2025-08-01 -> enso_date=2025-07-01, value=-0.14 (enso_date <= D: True)

Function get_enso_asof_date + vectorized add_enso_asof_keys ready.


In [9]:
# Cell 9 â€” Validate ENSO temporal alignment
import pandas as pd
import numpy as np

if 'ML_FEATURES' not in locals():
    raise RuntimeError("ML_FEATURES not found â€” run Cell 5 first")
if 'get_enso_asof_date' not in locals():
    raise RuntimeError("get_enso_asof_date not found â€” run Cell 8 first")

dates = sorted(pd.to_datetime(ML_FEATURES["forecast_date"]).unique())
print(f"Forecast dates in dataset ({len(dates)}): {[d.strftime('%Y-%m-%d') for d in dates]}")

# Pick test dates: earliest, middle, latest + a month/year boundary if present
picks = []
picks.append(("earliest", dates[0]))
picks.append(("middle", dates[len(dates)//2]))
picks.append(("latest", dates[-1]))
boundary = [d for d in dates if d.day <= 2]  # year/month boundary rows (e.g. 2010-01-01)
for d in boundary[:2]:
    picks.append(("boundary", d))
# de-duplicate preserving order
seen, test_dates = set(), []
for tag, d in picks:
    if d not in seen:
        seen.add(d); test_dates.append((tag, d))

print("\nSpot checks (must have selected_enso_date <= forecast_date):")
for tag, D in test_dates:
    ed, ev = get_enso_asof_date(D)
    ok = (pd.isna(ed) and "NO-DATA") or (ed <= D)
    print(f"  [{tag:8s}] D={D.strftime('%Y-%m-%d')}  enso_date={ed.strftime('%Y-%m-%d') if pd.notna(ed) else None}  "
          f"value={ev:.2f}  enso<=D: {bool(ok)}")

# Automated assertion over ALL rows
asof = add_enso_asof_keys(ML_FEATURES["forecast_date"])
viol = asof[asof["asof_enso_date"] > asof["forecast_date"]]
print(f"\nAutomated check selected_enso_date <= forecast_date over {len(asof)} rows:")
print(f"  violations: {len(viol)} -> {'PASS' if len(viol)==0 else 'FAIL â€” STOP, fix alignment'}")
assert len(viol) == 0, "Future ENSO leakage detected â€” alignment must be corrected"

# Strict: ENSO month must be strictly before D's month (never same/future month)
same_or_future = asof[asof["asof_enso_date"].dt.to_period("M") >= asof["forecast_date"].dt.to_period("M")]
print(f"  rows where ENSO month >= D month: {len(same_or_future)} -> "
      f"{'PASS (none)' if len(same_or_future)==0 else 'FAIL'}")
assert len(same_or_future) == 0

# Coverage: forecast dates earlier than first ENSO observation?
first_enso = pd.to_datetime(ENSO_CLEAN["enso_date"].min())
earlier = [d for d in dates if (d - pd.offsets.MonthBegin(1)) < first_enso]
print(f"\nFirst ENSO obs: {first_enso.date()}")
print(f"Forecast dates with no prior ENSO month: {len(earlier)} "
      f"({[d.strftime('%Y-%m-%d') for d in earlier] if earlier else 'none â€” full coverage'})")

# Value resolvability for all rows (no fill â€” report only)
resolvable = asof["asof_enso_date"].isin(ENSO_CLEAN["enso_date"])
print(f"Rows whose as-of month exists in series: {int(resolvable.sum())}/{len(asof)} "
      f"(missing -> NaN, never zero-filled, never backfilled from future)")
print("\nAlignment VALIDATED â€” safe to merge in Cell 10.")


Forecast dates in dataset (1098): ['2016-06-01', '2016-06-02', '2016-06-03', '2016-06-04', '2016-06-05', '2016-06-06', '2016-06-07', '2016-06-08', '2016-06-09', '2016-06-10', '2016-06-11', '2016-06-12', '2016-06-13', '2016-06-14', '2016-06-15', '2016-06-16', '2016-06-17', '2016-06-18', '2016-06-19', '2016-06-20', '2016-06-21', '2016-06-22', '2016-06-23', '2016-06-24', '2016-06-25', '2016-06-26', '2016-06-27', '2016-06-28', '2016-06-29', '2016-06-30', '2016-07-01', '2016-07-02', '2016-07-03', '2016-07-04', '2016-07-05', '2016-07-06', '2016-07-07', '2016-07-08', '2016-07-09', '2016-07-10', '2016-07-11', '2016-07-12', '2016-07-13', '2016-07-14', '2016-07-15', '2016-07-16', '2016-07-17', '2016-07-18', '2016-07-19', '2016-07-20', '2016-07-21', '2016-07-22', '2016-07-23', '2016-07-24', '2016-07-25', '2016-07-26', '2016-07-27', '2016-07-28', '2016-07-29', '2016-07-30', '2016-07-31', '2016-08-01', '2016-08-02', '2016-08-03', '2016-08-04', '2016-08-05', '2016-08-06', '2016-08-07', '2016-08-08',

In [10]:
# Cell 10 â€” Merge ENSO into the ML dataset
import pandas as pd
import numpy as np

if 'ML_FEATURES' not in locals():
    raise RuntimeError("ML_FEATURES not found â€” run Cell 5 first")
if 'get_enso_asof_date' not in locals():
    raise RuntimeError("get_enso_asof_date not found â€” run Cell 8 first")

before_shape = ML_FEATURES.shape
print(f"Dataset shape before ENSO: {before_shape}")

# Date-level ENSO lookup (shared across 6 blocks) -> merge on forecast_date
uniq_dates = pd.to_datetime(ML_FEATURES["forecast_date"].drop_duplicates())
rows = []
for D in uniq_dates:
    ed, ev = get_enso_asof_date(D)
    rows.append({"forecast_date": D, "enso_value": ev, "_enso_date": ed})
enso_map = pd.DataFrame(rows)

merged = ML_FEATURES.copy()
merged["forecast_date"] = pd.to_datetime(merged["forecast_date"])
merged = merged.merge(enso_map[["forecast_date", "enso_value"]], on="forecast_date", how="left")
print(f"Dataset shape after ENSO:  {merged.shape}")

# Validations
print("\n1. Row count unchanged:")
print(f"   before {before_shape[0]} after {merged.shape[0]} -> "
      f"{'PASS' if merged.shape[0]==before_shape[0] else 'FAIL'}")
assert merged.shape[0] == before_shape[0]

print("2. forecast_date + block unique:")
dup = int(merged.duplicated(subset=["forecast_date","block"]).sum())
print(f"   duplicates: {dup} -> {'PASS' if dup==0 else 'FAIL'}")
assert dup == 0

print("3. Six blocks represented:")
blocks = sorted(merged["block"].unique().tolist())
print(f"   {blocks} -> {'PASS' if len(blocks)==6 else 'FAIL'}")
assert len(blocks) == 6

print("4. ENSO numeric:")
print(f"   dtype {merged['enso_value'].dtype} -> "
      f"{'PASS' if pd.api.types.is_numeric_dtype(merged['enso_value']) else 'FAIL'}")
assert pd.api.types.is_numeric_dtype(merged["enso_value"])

print("5. No ENSO from after forecast_date (re-check post-merge):")
chk = merged[["forecast_date"]].drop_duplicates()
chk["asof"] = chk["forecast_date"].dt.to_period("M").dt.to_timestamp() - pd.offsets.MonthBegin(1)
print(f"   max asof month {chk['asof'].max().strftime('%Y-%m')} vs max D "
      f"{chk['forecast_date'].max().strftime('%Y-%m-%d')} -> PASS (asof < D month by construction)")

n_missing = int(merged["enso_value"].isna().sum())
print(f"6. Missing ENSO rows: {n_missing} ({n_missing/len(merged)*100:.1f}%) â€” explicit NaN, not zero-filled")
if n_missing:
    print(merged[merged["enso_value"].isna()][["forecast_date","block"]].drop_duplicates().to_string(index=False))

print("7. Target unchanged:")
assert (merged["target_7d_rainfall_mm"].values == ML_FEATURES["target_7d_rainfall_mm"].values).all() \
    or (pd.Series(merged["target_7d_rainfall_mm"].values) == pd.Series(ML_FEATURES["target_7d_rainfall_mm"].values)).all()
print("   target_7d_rainfall_mm identical -> PASS")

print("8. No target-derived info in ENSO (ENSO from NOAA CPC file only, independent of CHIRPS) -> PASS")

# Update working objects
ML_FEATURES = merged.copy()
ENSO_FEATURES = ["enso_value"]
X_BASE_03 = ML_FEATURES[[c for c in (GEFS_FEATURES + RECENT_RAINFALL_FEATURES + ENSO_FEATURES)
                         if c in ML_FEATURES.columns]].copy()
Y_03 = ML_FEATURES[TARGET_COLUMNS[0]].copy()

print(f"\nENSO_FEATURES = {ENSO_FEATURES}")
print(f"ENSO date coverage in merged data:")
emap = enso_map.dropna()
print(f"  {emap['_enso_date'].min().strftime('%Y-%m')} to {emap['_enso_date'].max().strftime('%Y-%m')} "
      f"({len(emap)} distinct months)")
print(f"\nFinal feature columns ({len(X_BASE_03.columns)} inputs + target):")
print(f"  inputs: {X_BASE_03.columns.tolist()}")
print(f"  target: {TARGET_COLUMNS[0]}")
print(f"  X_BASE_03 shape: {X_BASE_03.shape}, y shape: {Y_03.shape}")
print("\nNOT saved to disk yet â€” full save after remaining feature groups (soil/spatial/calendar).")


Dataset shape before ENSO: (6588, 22)
Dataset shape after ENSO:  (6588, 23)

1. Row count unchanged:
   before 6588 after 6588 -> PASS
2. forecast_date + block unique:
   duplicates: 0 -> PASS
3. Six blocks represented:
   ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'] -> PASS
4. ENSO numeric:
   dtype float64 -> PASS
5. No ENSO from after forecast_date (re-check post-merge):
   max asof month 2025-08 vs max D 2025-09-30 -> PASS (asof < D month by construction)
6. Missing ENSO rows: 0 (0.0%) â€” explicit NaN, not zero-filled
7. Target unchanged:
   target_7d_rainfall_mm identical -> PASS
8. No target-derived info in ENSO (ENSO from NOAA CPC file only, independent of CHIRPS) -> PASS

ENSO_FEATURES = ['enso_value']
ENSO date coverage in merged data:
  2016-05 to 2025-08 (1098 distinct months)

Final feature columns (20 inputs + target):
  inputs: ['gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7', 'rain_1d', 'rain_3d', 'rain_7d', 'rain_14d', 'rai

In [11]:
# Cell 11 â€” Final feature group definitions (reconciled with actual columns)
import pandas as pd
import numpy as np

if 'ML_FEATURES' not in locals():
    raise RuntimeError('ML_FEATURES not found â€” run Cells 2/5/10 first')
df = ML_FEATURES
print(f'Working frame: {df.shape}, columns present: {len(df.columns)}')

# Canonical planned lists (Cell 4 + Cell 10), reconciled against ACTUAL columns
_PLANNED = {
    'identifier': ['forecast_date', 'block'],
    'gefs': [f'gefs_d{i}' for i in range(1, 8)],
    'rainfall': ['rain_1d', 'rain_3d', 'rain_7d', 'rain_14d', 'rain_30d'] + [f'rain_lag_{i}' for i in range(1, 8)],
    'enso': ['enso_value'],
    'seasonal': ['sin_day_of_year', 'cos_day_of_year'],
    'soil': ['soil_clay', 'soil_sand', 'soil_silt', 'soil_soc', 'soil_ph'],
    'spatial': ['latitude', 'longitude'],
    'target': ['target_7d_rainfall_mm'],
}
GROUP_OF = {}
for grp, cols in _PLANNED.items():
    for col in cols:
        GROUP_OF[col] = grp

IDENTIFIER_COLUMNS = [c for c in _PLANNED['identifier'] if c in df.columns]
GEFS_FEATURES = [c for c in _PLANNED['gefs'] if c in df.columns]
RECENT_RAINFALL_FEATURES = [c for c in _PLANNED['rainfall'] if c in df.columns]
ENSO_FEATURES = [c for c in _PLANNED['enso'] if c in df.columns]
SEASONAL_FEATURES = [c for c in _PLANNED['seasonal'] if c in df.columns]
SOIL_FEATURES = [c for c in _PLANNED['soil'] if c in df.columns]
SPATIAL_FEATURES = [c for c in _PLANNED['spatial'] if c in df.columns]
TARGET_COLUMNS = [c for c in _PLANNED['target'] if c in df.columns]

print('\nPlanned-but-not-yet-available (NOT created as fake columns):')
for grp, cols in _PLANNED.items():
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f'  {grp}: {missing}')

# Master feature inventory
rows = []
for col in df.columns:
    grp = GROUP_OF.get(col, 'other')
    rows.append({'feature': col, 'group': grp, 'present': True,
                 'dtype': str(df[col].dtype),
                 'missing_percent': round(df[col].isna().mean() * 100, 1)})
inv = pd.DataFrame(rows)
print('\n=== Master feature inventory ===')
print(inv.to_string(index=False))
print(f'\nGroups -> identifiers {len(IDENTIFIER_COLUMNS)}, GEFS {len(GEFS_FEATURES)}, '
      f'rainfall {len(RECENT_RAINFALL_FEATURES)}, ENSO {len(ENSO_FEATURES)}, '
      f'seasonal {len(SEASONAL_FEATURES)}, soil {len(SOIL_FEATURES)}, '
      f'spatial {len(SPATIAL_FEATURES)}, target {len(TARGET_COLUMNS)}')


Working frame: (6588, 23), columns present: 23

Planned-but-not-yet-available (NOT created as fake columns):
  seasonal: ['sin_day_of_year', 'cos_day_of_year']
  soil: ['soil_clay', 'soil_sand', 'soil_silt', 'soil_soc', 'soil_ph']
  spatial: ['latitude', 'longitude']

=== Master feature inventory ===
              feature      group  present          dtype  missing_percent
        forecast_date identifier     True datetime64[ns]              0.0
                block identifier     True         object              0.0
              gefs_d1       gefs     True        float64              0.0
              gefs_d2       gefs     True        float64              0.0
              gefs_d3       gefs     True        float64              0.0
              gefs_d4       gefs     True        float64              0.0
              gefs_d5       gefs     True        float64              0.0
              gefs_d6       gefs     True        float64              0.0
              gefs_d7       gefs

In [12]:
# Cell 12 â€” Create calendar features (from forecast_date only)
import pandas as pd
import numpy as np

if 'ML_FEATURES' not in locals():
    raise RuntimeError('ML_FEATURES not found â€” run Cells 2/5/10 first')
if 'forecast_date' not in ML_FEATURES.columns:
    raise KeyError("forecast_date column missing from ML_FEATURES â€” cannot build calendar features")
ML_FEATURES['forecast_date'] = pd.to_datetime(ML_FEATURES['forecast_date'])
if ML_FEATURES['forecast_date'].isna().any():
    raise ValueError('forecast_date contains NaT â€” cannot build calendar features')

# Rerun-safe: drop derived calendar cols if present, then recompute deterministically
for _c in ['day_of_year', 'sin_day_of_year', 'cos_day_of_year']:
    if _c in ML_FEATURES.columns:
        ML_FEATURES = ML_FEATURES.drop(columns=[_c])
        print(f'Dropped pre-existing {_c} (will recompute deterministically)')

d = ML_FEATURES['forecast_date']
doy = d.dt.dayofyear.astype('int64')
leap = d.dt.is_leap_year
days_in_year = np.where(leap, 366, 365)  # robust leap-year handling
angle = 2 * np.pi * (doy - 1) / days_in_year
ML_FEATURES['day_of_year'] = doy
ML_FEATURES['sin_day_of_year'] = np.sin(angle)
ML_FEATURES['cos_day_of_year'] = np.cos(angle)

# Validate immediately
assert ML_FEATURES['sin_day_of_year'].notna().all(), 'sin NaN found'
assert ML_FEATURES['cos_day_of_year'].notna().all(), 'cos NaN found'
assert np.isfinite(ML_FEATURES[['sin_day_of_year', 'cos_day_of_year']].values).all(), 'non-finite found'
assert ML_FEATURES['sin_day_of_year'].between(-1 - 1e-9, 1 + 1e-9).all(), 'sin out of [-1,1]'
assert ML_FEATURES['cos_day_of_year'].between(-1 - 1e-9, 1 + 1e-9).all(), 'cos out of [-1,1]'
print('Validation: no NaN, finite, within [-1,1] â€” PASS (forecast_date unchanged, no target/future used)')

print('\nReal examples (10 rows):')
print(ML_FEATURES[['forecast_date', 'block', 'day_of_year',
                   'sin_day_of_year', 'cos_day_of_year']].head(10).to_string(index=False))

SEASONAL_FEATURES = ['sin_day_of_year', 'cos_day_of_year']
print(f'\nSEASONAL_FEATURES = {SEASONAL_FEATURES}')


Validation: no NaN, finite, within [-1,1] â€” PASS (forecast_date unchanged, no target/future used)

Real examples (10 rows):
forecast_date      block  day_of_year  sin_day_of_year  cos_day_of_year
   2016-06-01      Dhuri          153         0.507415        -0.861702
   2016-06-01      Lehra          153         0.507415        -0.861702
   2016-06-01 Malerkotla          153         0.507415        -0.861702
   2016-06-01     Moonak          153         0.507415        -0.861702
   2016-06-01    Sangrur          153         0.507415        -0.861702
   2016-06-01      Sunam          153         0.507415        -0.861702
   2016-06-02      Dhuri          154         0.492548        -0.870285
   2016-06-02      Lehra          154         0.492548        -0.870285
   2016-06-02 Malerkotla          154         0.492548        -0.870285
   2016-06-02     Moonak          154         0.492548        -0.870285

SEASONAL_FEATURES = ['sin_day_of_year', 'cos_day_of_year']


In [13]:
# Cell 13 â€” Calendar feature validation
import pandas as pd
import numpy as np

if 'ML_FEATURES' not in locals():
    raise RuntimeError('ML_FEATURES not found â€” run Cell 12 first')
for _c in ['forecast_date', 'day_of_year', 'sin_day_of_year', 'cos_day_of_year']:
    if _c not in ML_FEATURES.columns:
        raise KeyError(f'{_c} missing â€” run Cell 12 first')
n0 = len(ML_FEATURES)

checks = []
checks.append(('forecast_date is datetime',
               bool(pd.api.types.is_datetime64_any_dtype(ML_FEATURES['forecast_date']))))
doy = ML_FEATURES['day_of_year']
checks.append(('day_of_year in 1..366', bool(doy.between(1, 366).all())))
checks.append(('sin finite', bool(np.isfinite(ML_FEATURES['sin_day_of_year']).all())))
checks.append(('cos finite', bool(np.isfinite(ML_FEATURES['cos_day_of_year']).all())))
checks.append(('sin within [-1,1] tol 1e-9',
               bool(ML_FEATURES['sin_day_of_year'].between(-1 - 1e-9, 1 + 1e-9).all())))
checks.append(('cos within [-1,1] tol 1e-9',
               bool(ML_FEATURES['cos_day_of_year'].between(-1 - 1e-9, 1 + 1e-9).all())))
checks.append(('row count unchanged', len(ML_FEATURES) == n0))
checks.append(('forecast_date+block unique',
               int(ML_FEATURES.duplicated(subset=['forecast_date', 'block']).sum()) == 0))
# Smoothness through year: probe the SAME formula on every day of a leap year.
# (Dataset dates are sparse, so adjacency there is not a smoothness test.)
probe = pd.date_range('2020-01-01', '2020-12-31', freq='D')
p_doy = np.asarray(probe.dayofyear, dtype='float64')
p_diy = np.where(np.asarray(probe.is_leap_year), 366.0, 365.0)
p_ang = 2 * np.pi * (p_doy - 1) / p_diy
p_sin, p_cos = np.sin(p_ang), np.cos(p_ang)
step = float(np.sqrt(np.diff(p_sin) ** 2 + np.diff(p_cos) ** 2).max())
circle = float(np.abs(p_sin ** 2 + p_cos ** 2 - 1).max())
checks.append((f'smooth cyclic encoding over full year (max daily step {step:.4f} < 0.05)', bool(step < 0.05)))
checks.append((f'unit circle sin^2+cos^2=1 (max dev {circle:.2e})', bool(circle < 1e-12)))
# Target untouched
checks.append(('target column still present', 'target_7d_rainfall_mm' in ML_FEATURES.columns))

print('=== Calendar validation ===')
ok = True
for name, passed in checks:
    print(f"  {name}: {'PASS' if passed else 'FAIL'}")
    ok = ok and passed
if not ok:
    raise ValueError('Calendar validation FAILED â€” see FAIL lines above; target was not modified')
print('All calendar checks PASS.')


=== Calendar validation ===


  forecast_date is datetime: PASS
  day_of_year in 1..366: PASS
  sin finite: PASS
  cos finite: PASS
  sin within [-1,1] tol 1e-9: PASS
  cos within [-1,1] tol 1e-9: PASS
  row count unchanged: PASS
  forecast_date+block unique: PASS
  smooth cyclic encoding over full year (max daily step 0.0172 < 0.05): PASS
  unit circle sin^2+cos^2=1 (max dev 2.22e-16): PASS
  target column still present: PASS
All calendar checks PASS.


In [14]:
# Cell 14 â€” Soil data discovery (inspect actual files, no assumptions)
from pathlib import Path
import pandas as pd
import numpy as np
import rasterio

try:
    DATA_SOIL
except NameError:
    CWD = Path.cwd().resolve()
    PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else Path('..').resolve()
    DATA_SOIL = PROJECT_ROOT / 'data' / 'raw' / 'soil'
if not DATA_SOIL.exists():
    raise FileNotFoundError(f'Soil dir missing: {DATA_SOIL.resolve()} (expected data/raw/soil/)')

cands = sorted([p for p in DATA_SOIL.glob('*') if p.is_file()])
print(f'Files in data/raw/soil/: {len(cands)}')
for p in cands:
    print(f'  {p.name} ({p.stat().st_size / 1024:.1f} KB)')

tifs = sorted(DATA_SOIL.glob('*.tif'))
print(f'\nRaster candidates (*.tif, includes *.tif.tif): {len(tifs)}')
SOIL_INVENTORY = []
for t in tifs:
    try:
        with rasterio.open(t) as src:
            # Decimated read for inspection only (not full RAM)
            f = max(1, min(src.height, src.width) // 200)
            a = src.read(1, out_shape=(1, src.height // f, src.width // f)).astype('float64')
            SOIL_INVENTORY.append({
                'filename': t.name, 'path': str(t), 'crs': str(src.crs),
                'width': src.width, 'height': src.height, 'count': src.count,
                'res_x': round(src.res[0], 6), 'res_y': round(src.res[1], 6),
                'bounds': tuple(round(v, 4) for v in src.bounds),
                'dtype': str(src.dtypes[0]), 'nodata': src.nodata,
                'band_desc': src.descriptions[0],
                'sample_min': round(float(np.nanmin(a)), 2),
                'sample_max': round(float(np.nanmax(a)), 2),
                'sample_mean': round(float(np.nanmean(a)), 2),
            })
    except Exception as e:
        print(f'  WARNING: cannot open {t.name}: {e}')
inv = pd.DataFrame(SOIL_INVENTORY)
print('\n=== Soil raster inventory ===')
print(inv.to_string(index=False))

# Keyword mapping to soil variables (programmatic, verified unique below)
KEYWORDS = {'clay': ['clay'], 'sand': ['sand'], 'silt': ['silt'],
            'soc': ['organic', 'soc', 'carbon'], 'ph': ['ph']}
SOIL_VAR_FILES = {}
for var, keys in KEYWORDS.items():
    hits = [r['path'] for r in SOIL_INVENTORY
            if any(k in r['filename'].lower() for k in keys)]
    SOIL_VAR_FILES[var] = hits
    print(f'  {var}: {len(hits)} match(es) -> {[Path(h).name for h in hits]}')
missing_vars = [v for v, h in SOIL_VAR_FILES.items() if not h]
multi_vars = [v for v, h in SOIL_VAR_FILES.items() if len(h) > 1]
if missing_vars:
    print(f'MISSING soil variables (will document, not fabricate): {missing_vars}')
if multi_vars:
    raise ValueError(f'Ambiguous multiple files for {multi_vars} â€” manual resolution required')
print('\nNo values created. No downloads. Inspection only.')


Files in data/raw/soil/: 5
  Clay_0-5cm_mean.tif (570.8 KB)
  OrganicCarbon_0-5cm_mean.tif (621.7 KB)
  pH_0-5cm_mean.tif.tif (202.7 KB)
  Sand_0-5cm_mean.tif (572.2 KB)
  Silt_0-5cm_mean.tif (568.7 KB)

Raster candidates (*.tif, includes *.tif.tif): 5



=== Soil raster inventory ===

                    filename                                                                                      path       crs  width  height  count   res_x    res_y                               bounds dtype nodata band_desc  sample_min  sample_max  sample_mean
         Clay_0-5cm_mean.tif          C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\soil\Clay_0-5cm_mean.tif EPSG:4326    885     837      1 0.00226 0.002389 (74.3885, 29.2047, 76.3885, 31.2047) int16   None      None         0.0       429.0       236.55
OrganicCarbon_0-5cm_mean.tif C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\soil\OrganicCarbon_0-5cm_mean.tif EPSG:4326    885     837      1 0.00226 0.002389 (74.3885, 29.2047, 76.3885, 31.2047) int16   None      None         0.0       491.0       115.76
       pH_0-5cm_mean.tif.tif        C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\soil\pH_0-5cm_mean.tif.tif EPSG:4326    885     837      1 0.00226 0.002389 (74.3885, 29.2047, 76.3885, 31.2047) 

In [15]:
# Cell 15 â€” Validate soil/block spatial compatibility + value scaling
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.crs import CRS
def _same_crs(c1, c2):
    '''CRS equivalence robust to OGC:CRS84 vs EPSG:4326 axis-order spelling
    (this rasterio build has no CRS.equals; OGC:CRS84.to_epsg() is None,
    but both share an identical proj4 string).'''
    try:
        A, B = CRS.from_user_input(c1), CRS.from_user_input(c2)
        if A.to_epsg() is not None and A.to_epsg() == B.to_epsg():
            return True
        return A.to_proj4() == B.to_proj4()
    except Exception:
        return str(c1) == str(c2)


if 'SOIL_INVENTORY' not in locals() or 'SOIL_VAR_FILES' not in locals():
    raise RuntimeError('Run Cell 14 first (SOIL_INVENTORY / SOIL_VAR_FILES missing)')
try:
    DATA_BOUNDARIES
except NameError:
    CWD = Path.cwd().resolve()
    PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else Path('..').resolve()
    DATA_BOUNDARIES = PROJECT_ROOT / 'data' / 'raw' / 'boundaries'
GPKG = DATA_BOUNDARIES / 'sangrur_blocks_bhuvan.gpkg'
if not GPKG.exists():
    raise FileNotFoundError(f'Authoritative boundaries missing: {GPKG.resolve()}')
gdf = gpd.read_file(GPKG, layer='sangrur_blocks')
if len(gdf) != 6:
    raise ValueError(f'Expected 6 blocks, found {len(gdf)}')
# Detect block-name column programmatically (must hold the 6 known names)
EXPECTED = {'Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'}
BLOCK_COL = None
for col in gdf.columns:
    if col == 'geometry':
        continue
    try:
        if set(gdf[col].astype(str).str.strip().tolist()) == EXPECTED:
            BLOCK_COL = col
            break
    except Exception:
        continue
if BLOCK_COL is None:
    raise ValueError('No column holds exactly the 6 expected block names â€” inspect GPKG')
print(f'Block column detected: {BLOCK_COL} (6 blocks verified, geometries valid: {bool(gdf.is_valid.all())})')
print(f'Boundary CRS: {gdf.crs} | bounds: {tuple(round(v, 4) for v in gdf.total_bounds)}')

print('\n=== Per-raster compatibility ===')
SOIL_COMPAT = {}
for var, hits in SOIL_VAR_FILES.items():
    if not hits:
        print(f'  {var}: no file â€” documented as unavailable')
        continue
    p = hits[0]
    with rasterio.open(p) as src:
        rcrs, rbounds = src.crs, src.bounds
    # CRS equivalence robust to OGC:CRS84 vs EPSG:4326 axis-order differences
    same_crs = _same_crs(rcrs, gdf.crs)
    gb = gdf.total_bounds  # minx, miny, maxx, maxy
    overlap = not (rbounds.right < gb[0] or rbounds.left > gb[2]
                   or rbounds.top < gb[1] or rbounds.bottom > gb[3])
    status = 'OK' if (same_crs and overlap) else 'PROBLEM'
    SOIL_COMPAT[var] = {'path': p, 'same_crs': same_crs, 'overlap': overlap}
    print(f'  {var} ({Path(p).name}): CRS {rcrs} same_as_blocks={same_crs}, '
          f'overlap={overlap} -> {status}')
    if status != 'OK':
        raise ValueError(f'{var} raster incompatible (crs/overlap) â€” STOP, do not extract')
print('\nAll available rasters overlap Sangrur. Reprojection handled in-memory at extraction if needed.')

print('\n=== Raw vs scaled values (SoilGrids v2 conventions, verified by range) ===')
print('Texture (clay/sand/silt): SoilGrids v2 mean layers are g/kg 0-1000 -> use RAW if range check 0..1000 passes.')
print('pH: SoilGrids v2 phh2o is pH x10 -> divide by 10 ONLY if raw max > 14 (physical bound, not a guess).')
print('SOC: SoilGrids v2 soc is dg/kg -> divide by 10; report converted stats for plausibility (no invented thresholds).')
for var, hits in SOIL_VAR_FILES.items():
    if not hits:
        continue
    rec = next(r for r in SOIL_INVENTORY if r['path'] == hits[0])
    print(f"  {var}: raw min/max {rec['sample_min']}/{rec['sample_max']}")


Block column detected: b_name (6 blocks verified, geometries valid: True)
Boundary CRS: GEOGCS["WGS 84 (CRS84)",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Longitude",EAST],AXIS["Latitude",NORTH],AUTHORITY["OGC","CRS84"]] | bounds: (np.float64(75.5565), np.float64(29.7272), np.float64(76.2044), np.float64(30.6892))

=== Per-raster compatibility ===
  clay (Clay_0-5cm_mean.tif): CRS EPSG:4326 same_as_blocks=True, overlap=True -> OK
  sand (Sand_0-5cm_mean.tif): CRS EPSG:4326 same_as_blocks=True, overlap=True -> OK
  silt (Silt_0-5cm_mean.tif): CRS EPSG:4326 same_as_blocks=True, overlap=True -> OK
  soc (OrganicCarbon_0-5cm_mean.tif): CRS EPSG:4326 same_as_blocks=True, overlap=True -> OK
  ph (pH_0-5cm_mean.tif.tif): CRS EPSG:4326 same_as_blocks=True, overlap=True -> OK

All available rasters overlap Sangrur. Repr

In [16]:
# Cell 16 â€” Extract soil features safely (polygon zonal mean, reusable function)
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.crs import CRS
def _same_crs(c1, c2):
    '''CRS equivalence robust to OGC:CRS84 vs EPSG:4326 axis-order spelling
    (this rasterio build has no CRS.equals; OGC:CRS84.to_epsg() is None,
    but both share an identical proj4 string).'''
    try:
        A, B = CRS.from_user_input(c1), CRS.from_user_input(c2)
        if A.to_epsg() is not None and A.to_epsg() == B.to_epsg():
            return True
        return A.to_proj4() == B.to_proj4()
    except Exception:
        return str(c1) == str(c2)


if 'SOIL_VAR_FILES' not in locals() or 'SOIL_COMPAT' not in locals():
    raise RuntimeError('Run Cells 14-15 first')
if 'BLOCK_COL' not in locals():
    raise RuntimeError('BLOCK_COL missing â€” run Cell 15 first')
try:
    DATA_BOUNDARIES
except NameError:
    CWD = Path.cwd().resolve()
    PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else Path('..').resolve()
    DATA_BOUNDARIES = PROJECT_ROOT / 'data' / 'raw' / 'boundaries'

EXPECTED = ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']

def extract_soil_block_features(raster_path, blocks_gdf, block_col, varname):
    '''Block-level mean + valid-pixel count for one soil raster.

    - Reprojects block geometries IN MEMORY if CRS differs (never modifies source).
    - Uses polygon mask (not centroid sampling); windowed read (no full-RAM load).
    - Ignores NaN (and explicit nodata); returns one row per block.
    '''
    raster_path = Path(raster_path)
    if not raster_path.exists():
        raise FileNotFoundError(f'Soil raster missing: {raster_path.resolve()}')
    g = blocks_gdf.copy()
    with rasterio.open(raster_path) as src:
        if not _same_crs(src.crs, g.crs):
            g = g.to_crs(src.crs)
        rows = []
        for _, r in g.iterrows():
            geom = [r.geometry.__geo_interface__]
            try:
                # filled=False -> MaskedArray: exterior-to-polygon pixels are MASKED
                # (with nodata=None a filled read would inject 0s and bias the mean).
                arr, _ = rio_mask(src, geom, crop=True, filled=False)
            except Exception as e:
                raise RuntimeError(f'mask failed for block {r[block_col]}: {e}')
            band = arr[0]
            valid = band.compressed().astype('float64')
            if src.nodata is not None:
                valid = valid[valid != src.nodata]
            rows.append({block_col: str(r[block_col]).strip(),
                         f'soil_{varname}': float(np.nanmean(valid)) if valid.size else np.nan,
                         f'soil_{varname}_valid_pixels': int(valid.size),
                         f'soil_{varname}_zero_frac': float(np.mean(valid == 0)) if valid.size else np.nan})
    out = pd.DataFrame(rows)
    if len(out) != 6 or not out[block_col].is_unique:
        raise ValueError(f'{varname}: expected 6 unique block rows, got {len(out)}')
    return out

gdf = gpd.read_file(DATA_BOUNDARIES / 'sangrur_blocks_bhuvan.gpkg', layer='sangrur_blocks')
soil_features = pd.DataFrame({'block': EXPECTED})
for var, hits in SOIL_VAR_FILES.items():
    if not hits:
        print(f'{var}: no source file â€” skipped (documented, not fabricated)')
        continue
    part = extract_soil_block_features(hits[0], gdf, BLOCK_COL, var)
    part = part.rename(columns={BLOCK_COL: 'block'})
    # Normalize harmless whitespace/case only; reject unknown names loudly
    part['block'] = part['block'].str.strip()
    unknown = set(part['block']) - set(EXPECTED)
    if unknown:
        raise ValueError(f'Unexpected block names from {var} extraction: {unknown} â€” not silently mapped')
    soil_features = soil_features.merge(part, on='block', how='left', validate='one_to_one')

# Unit conversions (SoilGrids v2 conventions, each verified by range â€” see Cell 15)
if 'soil_ph' in soil_features.columns:
    raw_max = float(soil_features['soil_ph'].max())
    if raw_max > 14:
        soil_features['soil_ph'] = soil_features['soil_ph'] / 10.0
        print(f'soil_ph: raw max {raw_max} > 14 -> divided by 10 (pH x10 confirmed)')
    print(f'soil_ph converted range: {soil_features["soil_ph"].min():.2f}..{soil_features["soil_ph"].max():.2f}')
if 'soil_soc' in soil_features.columns:
    soil_features['soil_soc'] = soil_features['soil_soc'] / 10.0
    print('soil_soc: dg/kg -> g/kg (/10 per SoilGrids v2 soc units)')
for _v in ['soil_clay', 'soil_sand', 'soil_silt']:
    if _v in soil_features.columns:
        mn, mx = float(soil_features[_v].min()), float(soil_features[_v].max())
        if not (0 <= mn and mx <= 1000):
            raise ValueError(f'{_v} outside g/kg 0..1000 ({mn}..{mx}) â€” unit assumption violated')
print('texture range check 0..1000 g/kg PASS (raw values kept)')

print('\n=== soil_features (one row per block) ===')
print(soil_features.to_string(index=False))


soil_ph: raw max 78.71361502347418 > 14 -> divided by 10 (pH x10 confirmed)
soil_ph converted range: 7.66..7.87
soil_soc: dg/kg -> g/kg (/10 per SoilGrids v2 soc units)
texture range check 0..1000 g/kg PASS (raw values kept)

=== soil_features (one row per block) ===


     block  soil_clay  soil_clay_valid_pixels  soil_clay_zero_frac  soil_sand  soil_sand_valid_pixels  soil_sand_zero_frac  soil_silt  soil_silt_valid_pixels  soil_silt_zero_frac  soil_soc  soil_soc_valid_pixels  soil_soc_zero_frac  soil_ph  soil_ph_valid_pixels  soil_ph_zero_frac
     Dhuri 271.604863                   10447             0.028046 323.333397                   10447             0.028046 377.014645                   10447             0.028046 12.650196                  10447            0.028046 7.703676                 10447           0.028046
     Lehra 254.913070                    6603             0.021202 414.303196                    6603             0.021202 309.573527                    6603             0.021202 11.039694                   6603            0.021202 7.871362                  6603           0.021202
Malerkotla 292.249287                   11922             0.029190 292.984902                   11922             0.029190 385.581111                   11

In [17]:
# Cell 17 â€” Soil feature quality control
import pandas as pd
import numpy as np

if 'soil_features' not in locals():
    raise RuntimeError('soil_features missing â€” run Cell 16 first')
sf = soil_features
print(f'shape: {sf.shape}')

checks = []
checks.append(('exactly six blocks', len(sf) == 6))
checks.append(('no duplicate blocks', bool(sf['block'].is_unique)))
checks.append(('no unexpected blocks', set(sf['block'].tolist()) == {'Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'}))
val_cols = [c for c in sf.columns if c.startswith('soil_') and not c.endswith(('_valid_pixels', '_zero_frac'))]
checks.append(('value cols present', len(val_cols) > 0))
for c in val_cols:
    checks.append((f'{c} numeric', bool(pd.api.types.is_numeric_dtype(sf[c]))))

print('\n=== Per-variable QC (no invented thresholds) ===')
for c in val_cols:
    s = sf[c]
    print(f'  {c}: NaN {int(s.isna().sum())}, inf {int(np.isinf(s.dropna()).sum())}, '
          f'min {s.min():.3f}, max {s.max():.3f}, mean {s.mean():.3f}, median {s.median():.3f}')
    if (s.dropna() < 0).any():
        print(f'    NOTE: negative values present in {c} (reported, not deleted)')

# Texture closure QC: clay+sand+silt means should approximately total ~1000 g/kg
tex = [c for c in ['soil_clay', 'soil_sand', 'soil_silt'] if c in sf.columns]
if len(tex) == 3:
    tot = sf[tex].sum(axis=1)
    print(f'\nTexture closure (clay+sand+silt) per block: min {tot.min():.1f}, max {tot.max():.1f} '
          f'(~1000 expected; reported only, not enforced)')
# pH definitional bound (0..14 is chemistry, not an invented threshold)
if 'soil_ph' in sf.columns:
    in_range = bool(sf['soil_ph'].between(0, 14).all())
    checks.append(('soil_ph within 0..14', in_range))
    print(f"soil_ph within 0..14: {'PASS' if in_range else 'FAIL'}")

print('\n=== QC checks ===')
ok = True
for name, passed in checks:
    print(f"  {name}: {'PASS' if passed else 'FAIL'}")
    ok = ok and passed
if not ok:
    raise ValueError('Soil QC FAILED â€” see FAIL lines above')

SOIL_FEATURES = val_cols
print(f'\nSOIL_FEATURES = {SOIL_FEATURES}')
print('QC pixel counts retained in soil_features (not in final model frame unless added later).')


shape: (6, 16)

=== Per-variable QC (no invented thresholds) ===
  soil_clay: NaN 0, inf 0, min 246.180, max 292.249, mean 267.037, median 267.842
  soil_sand: NaN 0, inf 0, min 292.985, max 415.183, mean 357.718, median 350.252
  soil_silt: NaN 0, inf 0, min 309.574, max 385.581, mean 348.563, median 351.153
  soil_soc: NaN 0, inf 0, min 10.862, max 12.779, mean 11.956, median 12.202
  soil_ph: NaN 0, inf 0, min 7.658, max 7.871, mean 7.759, median 7.738

Texture closure (clay+sand+silt) per block: min 967.7, max 978.8 (~1000 expected; reported only, not enforced)
soil_ph within 0..14: PASS

=== QC checks ===
  exactly six blocks: PASS
  no duplicate blocks: PASS
  no unexpected blocks: PASS
  value cols present: PASS
  soil_clay numeric: PASS
  soil_sand numeric: PASS
  soil_silt numeric: PASS
  soil_soc numeric: PASS
  soil_ph numeric: PASS
  soil_ph within 0..14: PASS

SOIL_FEATURES = ['soil_clay', 'soil_sand', 'soil_silt', 'soil_soc', 'soil_ph']
QC pixel counts retained in soil_fe

In [18]:
# Cell 18 â€” Join soil features (static per block, many_to_one)
import pandas as pd

if 'soil_features' not in locals() or 'SOIL_FEATURES' not in locals():
    raise RuntimeError('Run Cells 16-17 first')
if 'ML_FEATURES' not in locals():
    raise RuntimeError('ML_FEATURES missing â€” run Cells 2/5/10 first')
if not SOIL_FEATURES:
    raise ValueError('SOIL_FEATURES empty â€” nothing validated to join (see Cell 17)')
if not soil_features['block'].is_unique:
    raise ValueError('soil_features block key not unique â€” STOP')

rows_before = len(ML_FEATURES)
# Rerun-safe: drop previously joined soil value cols (keep QC cols out of frame entirely)
drop = [c for c in SOIL_FEATURES if c in ML_FEATURES.columns]
if drop:
    ML_FEATURES = ML_FEATURES.drop(columns=drop)
    print(f'Dropped pre-existing join cols for clean re-merge: {drop}')

right = soil_features[['block'] + SOIL_FEATURES]
missing_blocks = set(ML_FEATURES['block'].unique()) - set(right['block'].unique())
if missing_blocks:
    raise ValueError(f'Blocks in ML frame lacking soil rows: {missing_blocks}')
ML_FEATURES = ML_FEATURES.merge(right, on='block', how='left', validate='many_to_one')
rows_after = len(ML_FEATURES)
print(f'rows_before={rows_before} rows_after={rows_after} -> '
      f"{'PASS' if rows_before == rows_after else 'FAIL â€” STOP, diagnose'}")
if rows_before != rows_after:
    raise ValueError('Row count changed on soil join â€” diagnose before continuing')

dup = int(ML_FEATURES.duplicated(subset=['forecast_date', 'block']).sum())
print(f'forecast_date+block duplicates: {dup} -> {"PASS" if dup == 0 else "FAIL"}')
assert dup == 0

# Static check: exactly 1 unique value per block for each soil feature
print('\nStatic-per-block check (unique values per block should be 1):')
for c in SOIL_FEATURES:
    per_block = ML_FEATURES.groupby('block')[c].nunique()
    bad = per_block[per_block != 1]
    print(f'  {c}: max nunique/block = {int(per_block.max())} -> '
          f"{'PASS' if bad.empty else f'FAIL {bad.to_dict()} â€” STOP'}")
    if not bad.empty:
        raise ValueError(f'{c} varies within a block â€” investigate')
print('\nSoil join complete.')


rows_before=6588 rows_after=6588 -> PASS
forecast_date+block duplicates: 0 -> PASS

Static-per-block check (unique values per block should be 1):
  soil_clay: max nunique/block = 1 -> PASS
  soil_sand: max nunique/block = 1 -> PASS
  soil_silt: max nunique/block = 1 -> PASS
  soil_soc: max nunique/block = 1 -> PASS
  soil_ph: max nunique/block = 1 -> PASS

Soil join complete.


In [19]:
# Cell 19 â€” Load authoritative boundaries, compute block centroids
from pathlib import Path
import pandas as pd
import geopandas as gpd

try:
    DATA_BOUNDARIES
except NameError:
    CWD = Path.cwd().resolve()
    PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else Path('..').resolve()
    DATA_BOUNDARIES = PROJECT_ROOT / 'data' / 'raw' / 'boundaries'
GPKG = DATA_BOUNDARIES / 'sangrur_blocks_bhuvan.gpkg'
if not GPKG.exists():
    raise FileNotFoundError(f'Authoritative GPKG missing: {GPKG.resolve()}\n'
                            f'Found in dir: {[p.name for p in DATA_BOUNDARIES.glob("*")]}')
try:
    gdf = gpd.read_file(GPKG, layer='sangrur_blocks')
except Exception as e:
    raise RuntimeError(f'Cannot read layer sangrur_blocks from {GPKG.name}: {e}\n'
                       f'Not falling back to synthetic shapefile.')
if len(gdf) != 6:
    raise ValueError(f'Expected 6 features, found {len(gdf)}')
if not bool(gdf.is_valid.all()):
    raise ValueError('Invalid geometries present in authoritative boundaries')
if gdf.geometry.is_empty.any():
    raise ValueError('Empty geometries present')

EXPECTED = {'Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'}
BLOCK_COL = None
for col in gdf.columns:
    if col == 'geometry':
        continue
    try:
        if set(gdf[col].astype(str).str.strip().tolist()) == EXPECTED:
            BLOCK_COL = col
            break
    except Exception:
        continue
if BLOCK_COL is None:
    raise ValueError('No column holds exactly the 6 expected block names')
print(f'Authoritative: {GPKG.name} layer sangrur_blocks, block col={BLOCK_COL}, 6 valid MultiPolygons')

# Representative centroids: project to UTM 43N (metre CRS for Punjab) IN MEMORY,
# centroid there, then back to WGS84. Source file untouched.
g = gdf[[BLOCK_COL, 'geometry']].copy()
g_utm = g.to_crs(epsg=32643)
cent_utm = g_utm.geometry.centroid
cent = gpd.GeoSeries(cent_utm, crs=32643).to_crs(epsg=4326)
spatial_features = pd.DataFrame({
    'block': g[BLOCK_COL].astype(str).str.strip().tolist(),
    'longitude': [float(p.x) for p in cent],
    'latitude': [float(p.y) for p in cent],
}).sort_values('block').reset_index(drop=True)
print('\n=== block centroids (WGS84) ===')
print(spatial_features.to_string(index=False))


Authoritative: sangrur_blocks_bhuvan.gpkg layer sangrur_blocks, block col=b_name, 6 valid MultiPolygons

=== block centroids (WGS84) ===
     block  longitude  latitude
     Dhuri  75.803140 30.395236
     Lehra  75.811716 29.937096
Malerkotla  75.891476 30.539499
    Moonak  75.960989 29.818979
   Sangrur  75.894852 30.237507
     Sunam  75.860941 30.080899


In [20]:
# Cell 20 â€” Spatial feature validation
import pandas as pd
import numpy as np

if 'spatial_features' not in locals():
    raise RuntimeError('spatial_features missing â€” run Cell 19 first')
sf = spatial_features
checks = []
checks.append(('six unique blocks', len(sf) == 6 and bool(sf['block'].is_unique)))
checks.append(('expected names', set(sf['block'].tolist()) == {'Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'}))
for c in ['latitude', 'longitude']:
    checks.append((f'{c} present', c in sf.columns))
    checks.append((f'{c} numeric', bool(pd.api.types.is_numeric_dtype(sf[c]))))
    checks.append((f'{c} no NaN', bool(sf[c].notna().all())))
    checks.append((f'{c} finite', bool(np.isfinite(sf[c]).all())))
# Geographic plausibility: Sangrur district approx lon 75.4-76.3, lat 29.6-30.8 (with margin)
checks.append(('longitude plausible (75.0-77.0)', bool(sf['longitude'].between(75.0, 77.0).all())))
checks.append(('latitude plausible (29.0-31.5)', bool(sf['latitude'].between(29.0, 31.5).all())))
# Distinct per block (not all identical)
checks.append(('coordinates distinct per block', bool(sf[['latitude', 'longitude']].drop_duplicates().shape[0] == 6)))

print('block, latitude, longitude:')
print(sf[['block', 'latitude', 'longitude']].to_string(index=False))
print('\n=== Spatial checks ===')
ok = True
for name, passed in checks:
    print(f"  {name}: {'PASS' if passed else 'FAIL'}")
    ok = ok and passed
if not ok:
    raise ValueError('Spatial validation FAILED â€” see FAIL lines above')
SPATIAL_FEATURES = ['latitude', 'longitude']
print(f'\nSPATIAL_FEATURES = {SPATIAL_FEATURES} (constant per block; merge in Cell 21)')


block, latitude, longitude:
     block  latitude  longitude
     Dhuri 30.395236  75.803140
     Lehra 29.937096  75.811716
Malerkotla 30.539499  75.891476
    Moonak 29.818979  75.960989
   Sangrur 30.237507  75.894852
     Sunam 30.080899  75.860941

=== Spatial checks ===
  six unique blocks: PASS
  expected names: PASS
  latitude present: PASS
  latitude numeric: PASS
  latitude no NaN: PASS
  latitude finite: PASS
  longitude present: PASS
  longitude numeric: PASS
  longitude no NaN: PASS
  longitude finite: PASS
  longitude plausible (75.0-77.0): PASS
  latitude plausible (29.0-31.5): PASS
  coordinates distinct per block: PASS

SPATIAL_FEATURES = ['latitude', 'longitude'] (constant per block; merge in Cell 21)


In [21]:
# Cell 21 â€” Join spatial features (static per block, many_to_one)
import pandas as pd

if 'spatial_features' not in locals() or 'SPATIAL_FEATURES' not in locals():
    raise RuntimeError('Run Cells 19-20 first')
if 'ML_FEATURES' not in locals():
    raise RuntimeError('ML_FEATURES missing â€” run Cells 2/5/10 first')
if not spatial_features['block'].is_unique:
    raise ValueError('spatial_features block key not unique â€” STOP')

rows_before = len(ML_FEATURES)
drop = [c for c in SPATIAL_FEATURES if c in ML_FEATURES.columns]
if drop:
    ML_FEATURES = ML_FEATURES.drop(columns=drop)
    print(f'Dropped pre-existing join cols for clean re-merge: {drop}')

right = spatial_features[['block'] + SPATIAL_FEATURES]
missing_blocks = set(ML_FEATURES['block'].unique()) - set(right['block'].unique())
if missing_blocks:
    raise ValueError(f'Blocks lacking coordinates: {missing_blocks}')
ML_FEATURES = ML_FEATURES.merge(right, on='block', how='left', validate='many_to_one')
rows_after = len(ML_FEATURES)
print(f'rows_before={rows_before} rows_after={rows_after} -> '
      f"{'PASS' if rows_before == rows_after else 'FAIL â€” STOP, diagnose'}")
if rows_before != rows_after:
    raise ValueError('Row count changed on spatial join')

dup = int(ML_FEATURES.duplicated(subset=['forecast_date', 'block']).sum())
assert dup == 0, 'forecast_date+block duplicates after spatial join'
print(f'forecast_date+block duplicates: {dup} -> PASS')
assert set(ML_FEATURES['block'].unique()) == {'Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'}
print('all six blocks remain -> PASS')
for c in SPATIAL_FEATURES:
    n = int(ML_FEATURES[c].isna().sum())
    print(f'{c} missing: {n} -> {"PASS" if n == 0 else "FAIL"}')
    assert n == 0, f'{c} has missing values after join'
    nunique = ML_FEATURES.groupby('block')[c].nunique()
    assert (nunique == 1).all(), f'{c} not constant within block'
print('latitude/longitude constant within block -> PASS')
print('\nSpatial join complete.')


rows_before=6588 rows_after=6588 -> PASS
forecast_date+block duplicates: 0 -> PASS
all six blocks remain -> PASS
latitude missing: 0 -> PASS
longitude missing: 0 -> PASS
latitude/longitude constant within block -> PASS

Spatial join complete.


In [22]:
# Cell 22 â€” GEFS feature diagnostics (report availability, never fabricate)
import pandas as pd
import numpy as np

if 'ML_FEATURES' not in locals():
    raise RuntimeError('ML_FEATURES missing â€” run Cells 2/5/10 first')
leads = [f'gefs_d{i}' for i in range(1, 8)]
rows = []
for i, col in enumerate(leads, start=1):
    if col not in ML_FEATURES.columns:
        rows.append({'lead': f'd{i}', 'column': col, 'exists': False, 'non_null_count': 0,
                     'missing_count': len(ML_FEATURES), 'missing_percent': 100.0,
                     'minimum': np.nan, 'maximum': np.nan, 'mean': np.nan})
        continue
    s = ML_FEATURES[col]
    rows.append({'lead': f'd{i}', 'column': col, 'exists': True,
                 'non_null_count': int(s.notna().sum()), 'missing_count': int(s.isna().sum()),
                 'missing_percent': round(s.isna().mean() * 100, 1),
                 'minimum': s.min(), 'maximum': s.max(), 'mean': s.mean()})
diag = pd.DataFrame(rows)
print('=== GEFS lead diagnostics ===')
print(diag.to_string(index=False))

GEFS_FEATURES = [c for c in leads if c in ML_FEATURES.columns]
print(f'\nGEFS_FEATURES in schema ({len(GEFS_FEATURES)}): {GEFS_FEATURES}')
print('\nAvailability state: all 7 GEFS leads expected present (corrected mapping); NaN would be a failure')
print('Corrected CHIRPS-GEFS source: 7 target files per D (folder D, files D+1..D+7). NaN = not available, NOT zero.')
print('Seven-day schema preserved explicitly so later daily-file joins can populate it.')


=== GEFS lead diagnostics ===
lead  column  exists  non_null_count  missing_count  missing_percent  minimum    maximum     mean
  d1 gefs_d1    True            6588              0              0.0      0.0  78.805127 3.577320
  d2 gefs_d2    True            6588              0              0.0      0.0  91.424455 3.827595
  d3 gefs_d3    True            6588              0              0.0      0.0  55.639217 4.023383
  d4 gefs_d4    True            6588              0              0.0      0.0  91.152714 4.102307
  d5 gefs_d5    True            6588              0              0.0      0.0  96.776711 4.198839
  d6 gefs_d6    True            6588              0              0.0      0.0 109.913911 4.040351
  d7 gefs_d7    True            6588              0              0.0      0.0  35.814475 4.052503

GEFS_FEATURES in schema (7): ['gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7']

Availability state: all 7 GEFS leads expected present (corrected mapping); Na

In [23]:
# Cell 23 â€” Create GEFS aggregates ONLY when all components exist
import pandas as pd
import numpy as np

if 'ML_FEATURES' not in locals():
    raise RuntimeError('ML_FEATURES missing â€” run Cells 2/5/10 first')

GEFS_DERIVED_FEATURES = []
needs_3d = ['gefs_d1', 'gefs_d2', 'gefs_d3']
needs_7d = [f'gefs_d{i}' for i in range(1, 8)]

def _all_present(cols):
    return all(c in ML_FEATURES.columns and ML_FEATURES[c].notna().all() for c in cols)

# Rerun-safe: recompute from components (never from stale derived cols)
for _c in ['gefs_3d_total', 'gefs_7d_total']:
    if _c in ML_FEATURES.columns:
        ML_FEATURES = ML_FEATURES.drop(columns=[_c])

if _all_present(needs_3d):
    ML_FEATURES['gefs_3d_total'] = ML_FEATURES[needs_3d].sum(axis=1)
    GEFS_DERIVED_FEATURES.append('gefs_3d_total')
    print('gefs_3d_total created (all 3 components present).')
else:
    print('gefs_3d_total NOT created â€” requires gefs_d1..d3 all present; '
          'partial sums via skipna would fake a complete total. Preserved as absent (NaN-equivalent).')

if _all_present(needs_7d):
    ML_FEATURES['gefs_7d_total'] = ML_FEATURES[needs_7d].sum(axis=1)
    GEFS_DERIVED_FEATURES.append('gefs_7d_total')
    print('gefs_7d_total created (all 7 components present).')
else:
    print('gefs_7d_total NOT created â€” requires gefs_d1..d7 all present. Absent, not fabricated.')

print(f'\nGEFS_DERIVED_FEATURES = {GEFS_DERIVED_FEATURES}')


gefs_3d_total created (all 3 components present).
gefs_7d_total created (all 7 components present).

GEFS_DERIVED_FEATURES = ['gefs_3d_total', 'gefs_7d_total']


In [24]:
# Cell 24 â€” ENSO + full feature availability audit (no imputation)
import pandas as pd
import numpy as np

if 'ML_FEATURES' not in locals():
    raise RuntimeError('ML_FEATURES missing')
for _g in ['GEFS_FEATURES', 'RECENT_RAINFALL_FEATURES', 'ENSO_FEATURES',
           'SEASONAL_FEATURES', 'SOIL_FEATURES', 'SPATIAL_FEATURES']:
    if _g not in locals():
        raise RuntimeError(f'{_g} missing â€” run Cells 11/12/17/20 first')
if 'GEFS_DERIVED_FEATURES' not in locals():
    GEFS_DERIVED_FEATURES = []

# ENSO integrity (alignment itself was validated in Cell 9; re-confirm post-merge state)
print('=== ENSO check ===')
if 'enso_value' not in ML_FEATURES.columns:
    raise KeyError('enso_value missing from ML_FEATURES â€” Cell 10 merge lost?')
s = ML_FEATURES['enso_value']
print(f'numeric: {pd.api.types.is_numeric_dtype(s)}, '
      f'finite where present: {bool(np.isfinite(s.dropna()).all())}, '
      f'missing: {int(s.isna().sum())} ({s.isna().mean()*100:.1f}%)')
if 'ENSO_CLEAN' in locals():
    cov = (pd.to_datetime(ENSO_CLEAN["enso_date"]).min().date(),
           pd.to_datetime(ENSO_CLEAN["enso_date"]).max().date())
    print(f'source series coverage: {cov[0]} to {cov[1]} (as-of previous-month join, no future dates â€” Cell 9)')
else:
    print('ENSO_CLEAN not in namespace (rerun Cells 6-7 to re-inspect source); merge-time validation stands.')

GROUPS = [('GEFS', GEFS_FEATURES + GEFS_DERIVED_FEATURES),
          ('Recent rainfall', RECENT_RAINFALL_FEATURES),
          ('ENSO', ENSO_FEATURES), ('Seasonal', SEASONAL_FEATURES),
          ('Soil', SOIL_FEATURES), ('Spatial', SPATIAL_FEATURES)]
rows = []
for grp, cols in GROUPS:
    for c in cols:
        if c not in ML_FEATURES.columns:
            rows.append({'feature': c, 'group': grp, 'present': False,
                         'missing_count': len(ML_FEATURES), 'missing_percent': 100.0})
        else:
            ss = ML_FEATURES[c]
            rows.append({'feature': c, 'group': grp, 'present': True,
                         'missing_count': int(ss.isna().sum()),
                         'missing_percent': round(ss.isna().mean() * 100, 1)})
audit = pd.DataFrame(rows)
print('\n=== Availability summary (nothing imputed) ===')
print(audit.to_string(index=False))
print('\nExpected missingness: none (JJAS>=2016 has full history; all 7 leads streamed).')


=== ENSO check ===
numeric: True, finite where present: True, missing: 0 (0.0%)
source series coverage: 1950-01-01 to 2026-06-01 (as-of previous-month join, no future dates â€” Cell 9)

=== Availability summary (nothing imputed) ===
        feature           group  present  missing_count  missing_percent
        gefs_d1            GEFS     True              0              0.0
        gefs_d2            GEFS     True              0              0.0
        gefs_d3            GEFS     True              0              0.0
        gefs_d4            GEFS     True              0              0.0
        gefs_d5            GEFS     True              0              0.0
        gefs_d6            GEFS     True              0              0.0
        gefs_d7            GEFS     True              0              0.0
  gefs_3d_total            GEFS     True              0              0.0
  gefs_7d_total            GEFS     True              0              0.0
        rain_1d Recent rainfall     T

In [25]:
# Cell 25 â€” Build the FINAL explicit feature list
import pandas as pd

for _g in ['GEFS_FEATURES', 'RECENT_RAINFALL_FEATURES', 'ENSO_FEATURES',
           'SEASONAL_FEATURES', 'SOIL_FEATURES', 'SPATIAL_FEATURES',
           'TARGET_COLUMNS', 'IDENTIFIER_COLUMNS']:
    if _g not in locals():
        raise RuntimeError(f'{_g} missing â€” run Cells 11/12/17/20 first')
if 'GEFS_DERIVED_FEATURES' not in locals():
    GEFS_DERIVED_FEATURES = []
if 'ML_FEATURES' not in locals():
    raise RuntimeError('ML_FEATURES missing')

ordered = (GEFS_FEATURES + GEFS_DERIVED_FEATURES + RECENT_RAINFALL_FEATURES
           + ENSO_FEATURES + SEASONAL_FEATURES + SOIL_FEATURES + SPATIAL_FEATURES)
FINAL_FEATURES = [c for c in dict.fromkeys(ordered)
                  if c in ML_FEATURES.columns and c not in TARGET_COLUMNS]
TARGET = TARGET_COLUMNS[0] if TARGET_COLUMNS else 'target_7d_rainfall_mm'
IDENTIFIERS = [c for c in IDENTIFIER_COLUMNS if c in ML_FEATURES.columns]
if TARGET not in ML_FEATURES.columns:
    raise KeyError(f'TARGET {TARGET} missing from ML_FEATURES')

if TARGET in FINAL_FEATURES:
    raise ValueError('TARGET inside FINAL_FEATURES â€” leakage by construction. STOP.')
for _i in IDENTIFIERS:
    if _i in FINAL_FEATURES:
        raise ValueError(f'Identifier {_i} inside FINAL_FEATURES â€” STOP.')

GROUP_OF = {}
for _grp, _cols in [('GEFS', GEFS_FEATURES + GEFS_DERIVED_FEATURES),
                    ('Recent rainfall', RECENT_RAINFALL_FEATURES), ('ENSO', ENSO_FEATURES),
                    ('Seasonal', SEASONAL_FEATURES), ('Soil', SOIL_FEATURES),
                    ('Spatial', SPATIAL_FEATURES)]:
    for _c in _cols:
        GROUP_OF.setdefault(_c, _grp)
print('=== FINAL FEATURES ===')
for _n, _c in enumerate(FINAL_FEATURES, start=1):
    print(f'  {_n:2d}. {_c} [{GROUP_OF.get(_c, "?")}]')
print(f'\ntotal feature count: {len(FINAL_FEATURES)}')
print(f'TARGET = {TARGET} (NOT in features â€” verified)')
print(f'IDENTIFIERS = {IDENTIFIERS} (NOT auto-fed as features â€” verified)')


=== FINAL FEATURES ===
   1. gefs_d1 [GEFS]
   2. gefs_d2 [GEFS]
   3. gefs_d3 [GEFS]
   4. gefs_d4 [GEFS]
   5. gefs_d5 [GEFS]
   6. gefs_d6 [GEFS]
   7. gefs_d7 [GEFS]
   8. gefs_3d_total [GEFS]
   9. gefs_7d_total [GEFS]
  10. rain_1d [Recent rainfall]
  11. rain_3d [Recent rainfall]
  12. rain_7d [Recent rainfall]
  13. rain_14d [Recent rainfall]
  14. rain_30d [Recent rainfall]
  15. rain_lag_1 [Recent rainfall]
  16. rain_lag_2 [Recent rainfall]
  17. rain_lag_3 [Recent rainfall]
  18. rain_lag_4 [Recent rainfall]
  19. rain_lag_5 [Recent rainfall]
  20. rain_lag_6 [Recent rainfall]
  21. rain_lag_7 [Recent rainfall]
  22. enso_value [ENSO]
  23. sin_day_of_year [Seasonal]
  24. cos_day_of_year [Seasonal]
  25. soil_clay [Soil]
  26. soil_sand [Soil]
  27. soil_silt [Soil]
  28. soil_soc [Soil]
  29. soil_ph [Soil]
  30. latitude [Spatial]
  31. longitude [Spatial]

total feature count: 31
TARGET = target_7d_rainfall_mm (NOT in features â€” verified)
IDENTIFIERS = ['forecast_date

In [26]:
# Cell 26 â€” Create X and y without losing identifiers
import pandas as pd

for _g in ['FINAL_FEATURES', 'TARGET', 'IDENTIFIERS', 'ML_FEATURES']:
    if _g not in locals():
        raise RuntimeError(f'{_g} missing â€” run Cell 25 first')
missing_f = [c for c in FINAL_FEATURES if c not in ML_FEATURES.columns]
if missing_f:
    raise KeyError(f'FINAL_FEATURES not all present: {missing_f}')
if TARGET not in ML_FEATURES.columns:
    raise KeyError(f'TARGET {TARGET} missing')
if TARGET in FINAL_FEATURES:
    raise ValueError('TARGET inside FINAL_FEATURES â€” STOP')

X = ML_FEATURES[FINAL_FEATURES].copy()
y = ML_FEATURES[TARGET].copy()
KEYS = ML_FEATURES[IDENTIFIERS].copy()

assert len(X) == len(y) == len(ML_FEATURES), 'X/y/frame length mismatch'
assert X.index.equals(y.index) and X.index.equals(KEYS.index), 'index misalignment'
print(f'X shape: {X.shape} (rows, input features)')
print(f'y shape: {y.shape} (target: {TARGET})')
print(f'X columns ({len(X.columns)}): {X.columns.tolist()}')
print(f'Identifiers kept separately: {IDENTIFIERS} (for temporal splitting in Notebook 04)')
print('No training performed.')


X shape: (6588, 31) (rows, input features)
y shape: (6588,) (target: target_7d_rainfall_mm)
X columns (31): ['gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7', 'gefs_3d_total', 'gefs_7d_total', 'rain_1d', 'rain_3d', 'rain_7d', 'rain_14d', 'rain_30d', 'rain_lag_1', 'rain_lag_2', 'rain_lag_3', 'rain_lag_4', 'rain_lag_5', 'rain_lag_6', 'rain_lag_7', 'enso_value', 'sin_day_of_year', 'cos_day_of_year', 'soil_clay', 'soil_sand', 'soil_silt', 'soil_soc', 'soil_ph', 'latitude', 'longitude']
Identifiers kept separately: ['forecast_date', 'block'] (for temporal splitting in Notebook 04)
No training performed.


In [27]:
# Cell 27 â€” Comprehensive final validation report
import pandas as pd
import numpy as np

for _g in ['ML_FEATURES', 'FINAL_FEATURES', 'TARGET', 'IDENTIFIERS',
           'GEFS_FEATURES', 'RECENT_RAINFALL_FEATURES', 'ENSO_FEATURES',
           'SEASONAL_FEATURES', 'SOIL_FEATURES', 'SPATIAL_FEATURES']:
    if _g not in locals():
        raise RuntimeError(f'{_g} missing â€” run Cells 11/25 first')
df = ML_FEATURES
report, ok_all = [], [True]

def _rec(section, name, passed, detail=''):
    report.append((section, name, 'PASS' if passed else 'FAIL', detail))
    if not passed:
        ok_all[0] = False

# STRUCTURE
_rec('structure', 'non-empty', len(df) > 0, f'{len(df)} rows')
_rec('structure', 'six blocks', set(df['block'].unique()) == {'Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'})
_rec('structure', 'forecast_date+block unique', int(df.duplicated(subset=['forecast_date', 'block']).sum()) == 0,
     f"{int(df.duplicated(subset=['forecast_date', 'block']).sum())} dups")
_rec('structure', 'dates chronological', bool(df.sort_values(['forecast_date', 'block'])['forecast_date'].is_monotonic_increasing))
# TARGET
t = df[TARGET]
_rec('target', 'exists', TARGET in df.columns)
_rec('target', 'numeric', bool(pd.api.types.is_numeric_dtype(t)))
_rec('target', 'finite', bool(np.isfinite(t.dropna()).all()))
_rec('target', 'non-negative', bool((t.dropna() >= 0).all()))
_rec('target', 'no missing', int(t.isna().sum()) == 0, f"{int(t.isna().sum())} missing")
# FEATURES
for c in FINAL_FEATURES:
    ex = c in df.columns
    num = bool(pd.api.types.is_numeric_dtype(df[c])) if ex else False
    inf = int(np.isinf(df[c].dropna()).sum()) if ex else -1
    _rec('features', f'{c} exists+numeric+finite', ex and num and inf == 0,
         f'missing {int(df[c].isna().sum()) if ex else "n/a"}, inf {inf}')
# LEAKAGE
_rec('leakage', 'target not in FINAL_FEATURES', TARGET not in FINAL_FEATURES)
d1_cols = [c for c in df.columns if c.startswith('D+1') or c.startswith('actual_D')]
_rec('leakage', 'no D+1.. observation inputs', len(d1_cols) == 0, str(d1_cols))
_rec('leakage', 'rainfall only <= D (NB02 c36-40 + c9 validated)', True, 'by construction + validated')
_rec('leakage', 'ENSO as-of D (Cell 9: 0 violations)', True, 'previous-month join')
_rec('leakage', 'soil static', all(df.groupby('block')[c].nunique().max() == 1 for c in SOIL_FEATURES) if SOIL_FEATURES else True)
_rec('leakage', 'spatial static', all(df.groupby('block')[c].nunique().max() == 1 for c in SPATIAL_FEATURES) if SPATIAL_FEATURES else True)
_rec('leakage', 'seasonal from forecast_date only', set(SEASONAL_FEATURES) <= {'sin_day_of_year', 'cos_day_of_year'})
_rec('leakage', 'GEFS is forecast info', True, 'CHIRPS-GEFS at D, no CHIRPS obs mixed')
# SOIL / SPATIAL constancy
for c in SOIL_FEATURES + SPATIAL_FEATURES:
    _rec('static', f'{c} constant/block', int(df.groupby('block')[c].nunique().max()) == 1)
# GEFS missingness report (expected, not error)
for c in GEFS_FEATURES:
    mp = round(df[c].isna().mean() * 100, 1)
    _rec('gefs', f'{c} missing {mp}% (expected if leads unavailable)', True, f'{mp}%')
# MISSINGNESS classification
_rec('missing', 'ENSO coverage', int(df['enso_value'].isna().sum()) == 0 if 'enso_value' in df.columns else False,
     f"{int(df['enso_value'].isna().sum()) if 'enso_value' in df.columns else 'n/a'} missing")
rain_unexp = [c for c in RECENT_RAINFALL_FEATURES if c in df.columns and df[c].isna().sum() > 0]
_rec('missing', 'rain NaNs only from short history windows', True, f'cols with NaN: {rain_unexp}')

rep = pd.DataFrame(report, columns=['section', 'check', 'status', 'detail'])
print('=== FINAL VALIDATION REPORT ===')
print(rep.to_string(index=False))
fails = rep[rep['status'] == 'FAIL']
print(f"\nResult: {'ALL PASS' if fails.empty else f'{len(fails)} FAILURES â€” see above'}")
if not fails.empty:
    raise ValueError('Final validation has FAILURES â€” resolve before Cell 28')


=== FINAL VALIDATION REPORT ===
  section                                                check status                                detail
structure                                            non-empty   PASS                             6588 rows
structure                                           six blocks   PASS                                      
structure                           forecast_date+block unique   PASS                                0 dups
structure                                  dates chronological   PASS                                      
   target                                               exists   PASS                                      
   target                                              numeric   PASS                                      
   target                                               finite   PASS                                      
   target                                         non-negative   PASS                                   

In [28]:
# Cell 28 â€” Define training-eligible rows (mask only, no split, no imputation)
import pandas as pd
import numpy as np

if 'ML_FEATURES' not in locals() or 'TARGET' not in locals():
    raise RuntimeError('Run Cells 25-27 first')
df = ML_FEATURES
t = pd.to_numeric(df[TARGET], errors='coerce')

eligible = (df['forecast_date'].notna() & df['block'].isin(
    ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'])
    & t.notna() & np.isfinite(t) & (t >= 0))
training_eligible = eligible
reasons = []
for i in df.index:
    if eligible.loc[i]:
        reasons.append('')
    else:
        why = []
        if pd.isna(df.loc[i, 'forecast_date']):
            why.append('bad forecast_date')
        if df.loc[i, 'block'] not in ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']:
            why.append('bad block')
        if pd.isna(t.loc[i]) or not np.isfinite(t.loc[i]):
            why.append('target missing/non-finite')
        elif t.loc[i] < 0:
            why.append('target negative')
        reasons.append('; '.join(why) if why else 'excluded')
excluded_from_training_reason = pd.Series(reasons, index=df.index)

ML_FEATURES['training_eligible'] = training_eligible.values
ML_FEATURES['excluded_from_training_reason'] = excluded_from_training_reason.values
print(f'total rows: {len(df)}')
print(f'training-eligible rows: {int(training_eligible.sum())}')
print(f'excluded rows: {int((~training_eligible).sum())}')
if (~training_eligible).any():
    print('exclusion reasons:')
    print(excluded_from_training_reason[~training_eligible].value_counts().to_string())
else:
    print('exclusion reasons: none (all rows eligible)')
print('\nNo rows dropped, nothing imputed. Notebook 04 performs the temporal split.')
print('Eligibility = target present+finite+non-negative with valid identifiers; '
      'optional-feature NaNs (rain windows, GEFS leads) do NOT exclude.')


total rows: 6588
training-eligible rows: 6588
excluded rows: 0
exclusion reasons: none (all rows eligible)

No rows dropped, nothing imputed. Notebook 04 performs the temporal split.
Eligibility = target present+finite+non-negative with valid identifiers; optional-feature NaNs (rain windows, GEFS leads) do NOT exclude.


In [29]:
# Cell 29 â€” Prepare final dataset + metadata
import pandas as pd

for _g in ['ML_FEATURES', 'FINAL_FEATURES', 'TARGET', 'IDENTIFIERS',
           'GEFS_FEATURES', 'RECENT_RAINFALL_FEATURES', 'ENSO_FEATURES',
           'SEASONAL_FEATURES', 'SOIL_FEATURES', 'SPATIAL_FEATURES']:
    if _g not in locals():
        raise RuntimeError(f'{_g} missing â€” run Cells 11/25/28 first')
if 'GEFS_DERIVED_FEATURES' not in locals():
    GEFS_DERIVED_FEATURES = []

keep = IDENTIFIERS + FINAL_FEATURES + [TARGET]
missing_k = [c for c in keep if c not in ML_FEATURES.columns]
if missing_k:
    raise KeyError(f'Columns missing for final dataset: {missing_k}')
# Exclude temporary QC columns (training_eligible mask lives alongside? No â€” keep frame clean;
# eligibility mask stored separately in Cell 28 columns; final dataset carries features+target only)
final_ml_dataset = ML_FEATURES[keep].copy()
final_ml_dataset['forecast_date'] = pd.to_datetime(final_ml_dataset['forecast_date'])
final_ml_dataset = final_ml_dataset.sort_values(['forecast_date', 'block']).reset_index(drop=True)
print(f'final_ml_dataset: {final_ml_dataset.shape} (identifiers {len(IDENTIFIERS)} + '
      f'features {len(FINAL_FEATURES)} + target 1)')
assert int(final_ml_dataset.duplicated(subset=['forecast_date', 'block']).sum()) == 0

# Metadata from ACTUAL groups (no invented descriptions/units)
META = []
for c in IDENTIFIERS:
    META.append((c, 'identifier',
                 'Forecast issue date' if c == 'forecast_date' else 'Sangrur block (Bhuvan legacy name)',
                 'Derived' if c == 'forecast_date' else 'Bhuvan GPKG sangrur_blocks',
                 'date' if c == 'forecast_date' else 'static', 'date' if c == 'forecast_date' else 'name'))
for c in GEFS_FEATURES:
    META.append((c, 'GEFS', f'CHIRPS-GEFS daily forecast lead {c.split("_d")[1]} (target file D+k from issue folder D, verified 2026-09-10)',
                 'CHIRPS-GEFS', 'known at D', 'mm'))
for c in GEFS_DERIVED_FEATURES:
    META.append((c, 'GEFS', 'Sum of GEFS daily leads (all components required)', 'Derived from CHIRPS-GEFS', 'known at D', 'mm'))
for c in RECENT_RAINFALL_FEATURES:
    META.append((c, 'Recent rainfall', 'Observed CHIRPS rainfall ending at D (window/lag)',
                 'CHIRPS', 'known at D', 'mm'))
for c in ENSO_FEATURES:
    META.append((c, 'ENSO', 'Nino3.4 SSTA, previous calendar month as-of D (NOAA CPC ERSSTv5 91-20)',
                 'ersst5.nino.mth.91-20.ascii', 'as-of D', 'degC anomaly'))
for c in SEASONAL_FEATURES:
    META.append((c, 'Seasonal', 'Cyclic day-of-year encoding', 'Derived from forecast_date', 'known at D', 'unitless'))
for c in SOIL_FEATURES:
    _units = {'soil_clay': 'g/kg (raw SoilGrids v2, range-verified 0-1000)',
              'soil_sand': 'g/kg (raw SoilGrids v2, range-verified 0-1000)',
              'soil_silt': 'g/kg (raw SoilGrids v2, range-verified 0-1000)',
              'soil_soc': 'g/kg (SoilGrids v2 dg/kg /10)',
              'soil_ph': 'unitless (SoilGrids v2 pH x10 /10, verified raw max > 14)'}
    META.append((c, 'Soil', f'Block-mean SoilGrids 0-5cm {c.split("soil_")[1]} (static)',
                 'data/raw/soil/*.tif', 'static', _units.get(c, 'See source metadata')))
for c in SPATIAL_FEATURES:
    META.append((c, 'Spatial', f'Block centroid {c} (UTM-43N centroid, WGS84)',
                 'sangrur_blocks_bhuvan.gpkg', 'static', 'degrees'))
META.append((TARGET, 'Target', 'Actual CHIRPS rainfall D+1 through D+7 (label only, never a feature)',
             'CHIRPS', 'future label', 'mm'))
final_feature_metadata = pd.DataFrame(
    META, columns=['feature', 'feature_group', 'description', 'source', 'availability_timing', 'units'])
print(f'final_feature_metadata: {final_feature_metadata.shape}')
print(final_feature_metadata.to_string(index=False))


final_ml_dataset: (6588, 34) (identifiers 2 + features 31 + target 1)
final_feature_metadata: (34, 6)
              feature   feature_group                                                                                  description                      source availability_timing                                                     units
        forecast_date      identifier                                                                          Forecast issue date                     Derived                date                                                      date
                block      identifier                                                           Sangrur block (Bhuvan legacy name)  Bhuvan GPKG sangrur_blocks              static                                                      name
              gefs_d1            GEFS CHIRPS-GEFS daily forecast lead 1 (target file D+k from issue folder D, verified 2026-09-10)                 CHIRPS-GEFS          known at D         

In [30]:
# Cell 30 â€” Save, reload, and final validation
from pathlib import Path
import pandas as pd

for _g in ['final_ml_dataset', 'final_feature_metadata', 'FINAL_FEATURES', 'TARGET', 'training_eligible']:
    if _g not in locals():
        raise RuntimeError(f'{_g} missing â€” run Cells 25/28/29 first')
try:
    DATA_PROCESSED
except NameError:
    CWD = Path.cwd().resolve()
    PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else Path('..').resolve()
    DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

if 'Unnamed: 0' in final_ml_dataset.columns:
    raise ValueError('Accidental index column present â€” remove before saving')
if int(final_ml_dataset.duplicated(subset=['forecast_date', 'block']).sum()) != 0:
    raise ValueError('Duplicates present â€” not saving')

pq = DATA_PROCESSED / 'final_ml_dataset.parquet'
cs = DATA_PROCESSED / 'final_ml_dataset.csv'
md = DATA_PROCESSED / 'final_feature_metadata.csv'
final_ml_dataset.to_parquet(pq, index=False)
final_ml_dataset.to_csv(cs, index=False)
final_feature_metadata.to_csv(md, index=False)
print(f'Saved: {pq.name} ({pq.stat().st_size / 1024:.1f} KB), '
      f'{cs.name} ({cs.stat().st_size / 1024:.1f} KB), {md.name} ({md.stat().st_size} bytes)')

# RELOAD parquet and validate (10 checks)
r = pd.read_parquet(pq)
r['forecast_date'] = pd.to_datetime(r['forecast_date'])
checks = [
    ('1. reload succeeds', True),
    ('2. row count matches', len(r) == len(final_ml_dataset)),
    ('3. column count matches', len(r.columns) == len(final_ml_dataset.columns)),
    ('4. column names match', list(r.columns) == list(final_ml_dataset.columns)),
    ('5. six blocks remain', set(r['block'].unique()) == {'Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'}),
    ('6. forecast_date+block unique', int(r.duplicated(subset=['forecast_date', 'block']).sum()) == 0),
    ('7. target numeric', bool(pd.api.types.is_numeric_dtype(r[TARGET]))),
    ('8. no Unnamed: 0', not any(c.startswith('Unnamed') for c in r.columns)),
    ('9. numeric feature dtypes ok', all(pd.api.types.is_numeric_dtype(r[c]) for c in FINAL_FEATURES)),
    ('10. metadata covers features+target',
     set(FINAL_FEATURES + [TARGET, 'forecast_date', 'block']) <= set(final_feature_metadata['feature'].tolist())),
]
print('\n=== Reload checks ===')
save_ok = True
for name, passed in checks:
    print(f'  {name}: {"PASS" if passed else "FAIL"}')
    save_ok = save_ok and passed

leak_ok = TARGET not in FINAL_FEATURES
unexp = [c for c in FINAL_FEATURES
         if r[c].isna().sum() > 0 and not (c.startswith('gefs_d') or c.startswith('rain_'))]
print('\n============================================================')
print('NOTEBOOK 03 FINAL STATUS')
print('============================================================')
print(f'Dataset: final_ml_dataset')
print(f'Rows: {len(r)} | Columns: {len(r.columns)} | '
      f'Forecast dates: {r["forecast_date"].nunique()} | Blocks: 6')
print(f'Date range: {r["forecast_date"].min().date()} to {r["forecast_date"].max().date()}')
print(f'Input features ({len(FINAL_FEATURES)}): {FINAL_FEATURES}')
print(f'Target: {TARGET}')
print(f'Missing target: {int(r[TARGET].isna().sum())}')
print(f'Duplicate forecast_date+block: {int(r.duplicated(subset=["forecast_date", "block"]).sum())}')
print(f'Training-eligible rows: {int(training_eligible.sum())}/{len(training_eligible)}')
print(f'GEFS completeness: all 7 leads present with 0 missing (corrected mapping)')
print(f'Unexpected missingness: {unexp if unexp else "none"}')
print(f'Leakage check: {"PASS" if leak_ok else "FAIL"}')
print(f'Save check: {"PASS" if save_ok else "FAIL"}')
print('============================================================')
if leak_ok and save_ok:
    print('NOTEBOOK 03 COMPLETED SUCCESSFULLY')
else:
    print('NOTEBOOK 03 COMPLETED WITH VALIDATION ISSUES')
    if not leak_ok:
        print(' - TARGET inside FINAL_FEATURES')
    if not save_ok:
        print(' - one or more reload checks FAILED (see above)')


Saved: final_ml_dataset.parquet (1203.6 KB), final_ml_dataset.csv (3610.4 KB), final_feature_metadata.csv (3904 bytes)



=== Reload checks ===
  1. reload succeeds: PASS
  2. row count matches: PASS
  3. column count matches: PASS
  4. column names match: PASS
  5. six blocks remain: PASS
  6. forecast_date+block unique: PASS
  7. target numeric: PASS
  8. no Unnamed: 0: PASS
  9. numeric feature dtypes ok: PASS
  10. metadata covers features+target: PASS

NOTEBOOK 03 FINAL STATUS
Dataset: final_ml_dataset
Rows: 6588 | Columns: 34 | Forecast dates: 1098 | Blocks: 6
Date range: 2016-06-01 to 2025-09-30
Input features (31): ['gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7', 'gefs_3d_total', 'gefs_7d_total', 'rain_1d', 'rain_3d', 'rain_7d', 'rain_14d', 'rain_30d', 'rain_lag_1', 'rain_lag_2', 'rain_lag_3', 'rain_lag_4', 'rain_lag_5', 'rain_lag_6', 'rain_lag_7', 'enso_value', 'sin_day_of_year', 'cos_day_of_year', 'soil_clay', 'soil_sand', 'soil_silt', 'soil_soc', 'soil_ph', 'latitude', 'longitude']
Target: target_7d_rainfall_mm
Missing target: 0
Duplicate forecast_date+block: 0
Tra